# Full replication notebook: SAVA → QLoRA Mastery Estimator → CCMF → Planner

This notebook rebuilds every experimental path used by the SoICT 2026 manuscript.

**Evidence streams**

| Stream | Label | Section |
|---|---|---|
| Zero-shot and QLoRA Mastery Estimator | REAL DATA / released LLMKT protocol | 6–9 |
| CCMF | REAL-LOGIT ABLATION (fit on validation, evaluated on untouched test) | 12 |
| SAVA | CONSTRUCTED BENCHMARK + REAL-TURN AGREEMENT | 10–11 |
| Planner (Maintain-Then-Advance) | SIMULATED DATA | 13 |
| Integrated online loop | REAL DATA (MathDial test turns) | 14 |
| Error Analyzer (LLM agent) | REAL DATA vs human gold (CoMTA) + SAVA-gated MathDial turns | 15 |
| Feedback Generator (LLM agent) | REAL DATA + automatic checks (+ optional LLM-as-judge) | 16 |

**CoMTA is used for evaluation only.** Its Khan Academy Evaluation Dataset License permits internal evaluation and prohibits model training, so with the default `COMTA_MODE = "evaluate_only"` no adapter, decision threshold or CCMF parameter is fitted on CoMTA: the MathDial adapter, the MathDial validation threshold and the MathDial CCMF parameters are evaluated on CoMTA. Set `COMTA_MODE = "train"` only with written permission from Khan Academy.

SAVA is **not** a preprocessing step for QLoRA training. QLoRA learns from released correctness labels. Online, the Mastery Estimator and CCMF predict the current turn first; SAVA then verifies the learner's answer and a non-abstaining verdict updates mastery only for subsequent turns.

**How to run**

1. `Runtime → Change runtime type → A100` (L4 also works, slower).
2. Keep `SMOKE = True` and choose `Runtime → Run all`. Paste your Hugging Face token when Section 4 asks. The smoke run exercises every cell on tiny subsets in roughly 20–30 minutes on A100; the first 16 GB model download dominates.
3. When the smoke run finishes without an assertion error, set `SMOKE = False` and `Run all` again for the full experiment.
4. Before running, add `OPENROUTER_API_KEY` in Colab Secrets (key icon in the left sidebar, enable *Notebook access*); it is used by the Section 16 judge and, if `PEDAGOGY_BACKEND = "openrouter"`, by the agents. Set `RUN_FEEDBACK_JUDGE = False` to run without OpenRouter.
5. Optional: upload `/content/error_analyzer_gold.csv` (human categories) before Section 15.
6. Trained adapters and the result ZIP are saved privately to Google Drive (`ARTIFACT_STORE = "drive"`; allow Drive access in Section 5b). To skip training on a later run, set `ADAPTER_SOURCE = "store"`.

## 1. Configuration

Change only the switches below. The run ID (with a `_smoke` suffix in smoke mode) prevents a rerun from overwriting earlier artifacts. Every artifact lives under `RUN_ROOT/{dataset}/{model}/{seed}/{split}/`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os, random, json, hashlib
import numpy as np
from IPython.display import display

SMOKE = True                 # True: end-to-end check on tiny subsets. Set False for the full run.
SEEDS = [221]                # QLoRA training seeds, e.g. [221, 222, 223] for a multi-seed study.
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
REPO_URL = "https://github.com/umass-ml4ed/dialogue-kt.git"
REPO_COMMIT = "c61f335f89005161b6ef439872cc1735bee26745"
DEPENDENCY_STACK = "pinned"  # "pinned": versions verified for this notebook; "latest": newest releases (golden test still guards drift).

ARTIFACT_STORE = "drive"     # private storage for adapters and result bundles: "drive" | "hub" | "none"
ADAPTER_SOURCE = "train"     # "train": run QLoRA training; "store": reuse adapters saved by an earlier run (skips training)
REUSE_RUN_ID = None          # with ADAPTER_SOURCE = "store": RUN_ID whose adapters to reuse; None = most recent match
DRIVE_ROOT = Path("/content/drive/MyDrive/soict2026_artifacts")
HF_ARTIFACT_REPO = None      # with ARTIFACT_STORE = "hub": e.g. "your-hf-name/soict2026-private-artifacts" (must stay private)

RUN_COMTA = True
RUN_MATHDIAL = True
COMTA_MODE = "evaluate_only"  # CoMTA license allows internal evaluation only: no adapter, threshold or CCMF parameter is
                              # trained on CoMTA (MathDial ones are transferred). "train" only with Khan Academy permission.
RUN_ZERO_SHOT = True         # Required for a matched QLoRA comparison.
RUN_QLORA = True
RUN_CCMF = True
CCMF_ON_ZERO_SHOT = True     # Also fit CCMF on zero-shot logits (tests calibration of uncalibrated outputs).
RUN_SAVA = True
RUN_PLANNER = True
RUN_INTEGRATED = True
RUN_ERROR_ANALYZER = True
RUN_FEEDBACK = True
PEDAGOGY_USE_KT_ADAPTER = False  # True: generate with the QLoRA KT adapter attached (ablation); False: base instruct model.
PEDAGOGY_BACKEND = "local"   # "local": Llama-3.1-8B on this GPU; "openrouter": PEDAGOGY_OPENROUTER_MODEL via OpenRouter.
PEDAGOGY_OPENROUTER_MODEL = "openai/gpt-4o-mini"
RUN_FEEDBACK_JUDGE = True    # LLM-as-judge through OpenRouter; needs OPENROUTER_API_KEY in Colab Secrets.
JUDGE_MODEL = "openai/gpt-4o"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

EXPORT_BATCH_SIZE = 1       # Same as the released test loop (apply_defaults sets batch_size=1).
SMOKE_DIALOGUES = 10         # The released --debug flag evaluates the first 10 test dialogues.
GOLDEN_AUC_TOL = 0.10        # Max |AUC_export - AUC_released| in percentage points.
CCMF_RESTARTS = 1 if SMOKE else 3
CCMF_MAXITER = 60 if SMOKE else 1500
N_BOOT = 50 if SMOKE else 1000
SAVA_N_REFERENCES = 30 if SMOKE else 454
SAVA_QUESTION_GUARD_GLOBAL = True
PLANNER_LEARNERS = 20 if SMOKE else 500
PLANNER_GRID_LEARNERS = 10 if SMOKE else 200
GEN_BATCH_SIZE = 8
GEN_MAX_NEW_TOKENS = 320
EA_MAX_COMTA_TURNS = 4 if SMOKE else None      # None: every incorrect CoMTA test turn
EA_MAX_MATHDIAL_TURNS = 4 if SMOKE else 200    # deterministic sample of SAVA-gated MathDial turns
FEEDBACK_CASES_PER_VERDICT = 1 if SMOKE else 10  # 3 verdicts x 10 = 30 cases

assert ARTIFACT_STORE in {"drive", "hub", "none"} and ADAPTER_SOURCE in {"train", "store"}
assert not (ADAPTER_SOURCE == "store" and ARTIFACT_STORE == "none"), "ADAPTER_SOURCE='store' needs an ARTIFACT_STORE."
STORE_PREFIX = "smoke" if SMOKE else "full"   # smoke adapters never mix with full-run adapters

PRIMARY_SEED = SEEDS[0]
RUN_ID =datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ("_smoke" if SMOKE else "")
WORK = Path("/content")
REPO = WORK / "dialogue-kt"
RUN_ROOT = WORK / "soict_framework_results" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONHASHSEED"] = str(PRIMARY_SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(PRIMARY_SEED)
np.random.seed(PRIMARY_SEED)

DATASETS = [name for name, enabled in [("comta", RUN_COMTA), ("mathdial", RUN_MATHDIAL)] if enabled]
assert COMTA_MODE in {"evaluate_only", "train"}
FIT_ON = {d: ("mathdial" if d == "comta" and COMTA_MODE == "evaluate_only" else d) for d in DATASETS}
assert all(source in DATASETS for source in FIT_ON.values()), "COMTA_MODE = 'evaluate_only' needs RUN_MATHDIAL = True."
FIT_ORDER = sorted(DATASETS, key=lambda d: FIT_ON[d] != d)   # datasets that fit their own parameters come first
MODEL_RUNS = ([("zero_shot", None)] if RUN_ZERO_SHOT else []) + ([("qlora", s) for s in SEEDS] if RUN_QLORA else [])

def seed_tag(model_key, seed):
    return "seed-na" if model_key == "zero_shot" else f"seed{seed}"

def run_dir(dataset, model_key, seed, split):
    return RUN_ROOT / dataset / model_key / seed_tag(model_key, seed) / split

print("Run ID:", RUN_ID, "| SMOKE:", SMOKE, "| seeds:", SEEDS)
print("Output:", RUN_ROOT)

## 2. Runtime and dependencies

The released `requirements.txt` (transformers 4.44.0 → tokenizers 0.19) has no wheels for the Python 3.13+ used by current Colab runtimes, so pip tries to compile `tokenizers` and fails. `DEPENDENCY_STACK = "pinned"` installs a current, mutually compatible stack instead (transformers 5.17.0, peft 0.20.0, accelerate 1.15.0, bitsandbytes 0.50.2, sentence-transformers 6.0.1) on top of Colab's preinstalled PyTorch. The golden test in Section 7 checks that the exported predictions still reproduce the released evaluation loop under these versions.

Do **not** add a cell that upgrades `pip`/`setuptools`: torch requires `setuptools<82`, which this cell enforces. If the cell reports that preinstalled packages changed, choose `Runtime → Restart session` and run all again. `pyBKT` and `pykt-toolkit` are required because `dialogue_kt/training.py` imports them at module level.

In [ ]:
import subprocess, sys
import importlib.metadata as metadata

print("Python", sys.version.split()[0])
def installed(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

preloaded = {p: installed(p) for p in ["torch", "numpy"]}
if DEPENDENCY_STACK == "pinned":
    core = ["transformers==5.17.0", "peft==0.20.0", "accelerate==1.15.0",
            "bitsandbytes==0.50.2", "sentence-transformers==6.0.1"]
else:
    core = ["-U", "transformers", "peft", "accelerate", "bitsandbytes", "sentence-transformers"]
# torch stays the Colab build; setuptools<82 repairs the torch requirement if an earlier cell upgraded it.
# krippendorff and openai are imported at module level by dialogue_kt.main (human_eval, annotate).
extras = ["setuptools<82", "sentencepiece", "pyBKT==1.4.3", "pykt-toolkit==0.0.38", "krippendorff", "openai",
          "scikit-learn", "scipy", "tqdm", "sympy", "matplotlib", "pandas"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *core, *extras], check=True)
changed = {p: (v, installed(p)) for p, v in preloaded.items() if installed(p) != v}
if changed:
    raise RuntimeError(f"Preinstalled packages changed {changed}. Choose Runtime > Restart session, then Run all again.")

import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU before continuing."
torch.manual_seed(PRIMARY_SEED)
torch.cuda.manual_seed_all(PRIMARY_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

import importlib.metadata as metadata
VERSIONS = {}
for package in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "sentence-transformers",
                "sympy", "scikit-learn", "scipy", "numpy", "pandas"]:
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = None
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu_name} ({gpu_gb:.2f} GB)")
print(json.dumps(VERSIONS, indent=2))
subprocess.run(["nvidia-smi"], check=True)

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, text=True, stdout=subprocess.PIPE).stdout
(RUN_ROOT / "pip_freeze.txt").write_text(freeze)
(RUN_ROOT / "runtime.json").write_text(json.dumps({"gpu": gpu_name, "gpu_memory_gb": round(gpu_gb, 2), **VERSIONS}, indent=2))

## 3. Obtain and patch the released implementation

The repository is pinned to the audited commit and tracked sources are restored before patching, so the cell is safe to rerun. Patches:

1. 8-bit → 4-bit NF4 loading and fp16 compute dtype (resource-efficient QLoRA).
2. `BCELoss` float cast (current PyTorch dtype compatibility).
3. `main.py` reads the seed from `DKT_SEED` instead of the hard-coded 221 (multi-seed runs). With the default `SEEDS = [221]` behaviour is unchanged.
4. `compute_metrics` returns `nan` AUC for a single-class subset instead of crashing (only reachable in smoke mode).

Prompts, splits, aggregation, labels and metric definitions are unchanged.

In [ ]:
import shutil

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--force", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--", "dialogue_kt"], check=True)

def patch(path, old, new, required=True):
    text = path.read_text()
    if old in text:
        path.write_text(text.replace(old, new))
    elif required and new not in text:
        raise RuntimeError(f"Patch anchor not found in {path}: {old[:60]!r}")

lm_path = REPO / "dialogue_kt/models/lm.py"
patch(lm_path,
      "bnb_config = BitsAndBytesConfig(\n    load_in_8bit=True,\n)",
      "bnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type=\"nf4\",\n"
      "    bnb_4bit_use_double_quant=True,\n    bnb_4bit_compute_dtype=torch.float16,\n)")
patch(lm_path,
      "torch_dtype=torch.float32 if quantize else torch.bfloat16,",
      "torch_dtype=torch.float16 if quantize else torch.bfloat16,")

training_path = REPO / "dialogue_kt/training.py"
patch(training_path,
      'torch.nn.BCELoss()(corr_probs, batch["labels"])',
      'torch.nn.BCELoss()(corr_probs.float(), batch["labels"].float())')
patch(training_path,
      "    auc = roc_auc_score(labels, preds)\n",
      "    auc = roc_auc_score(labels, preds) if len(set(labels)) > 1 else float(\"nan\")\n")

main_path = REPO / "dialogue_kt/main.py"
patch(main_path, "    initialize_seeds(221)",
      "    initialize_seeds(int(__import__(\"os\").environ.get(\"DKT_SEED\", \"221\")))")

try:
    import importlib.util
    pybkt_spec = importlib.util.find_spec("pyBKT")
    metrics_path = Path(next(iter(pybkt_spec.submodule_search_locations))) / "util/metrics.py"
    patch(metrics_path,
          "SUPPORTED_METRICS.update(fetch_supported_metrics())",
          "# Disabled for sklearn compatibility: SUPPORTED_METRICS.update(fetch_supported_metrics())",
          required=False)
except Exception as exc:
    print("pyBKT compatibility patch not needed or not applicable:", exc)

assert "load_in_4bit=True" in lm_path.read_text()
assert "corr_probs.float()" in training_path.read_text()
assert "DKT_SEED" in main_path.read_text()
(RUN_ROOT / "repository_commit.txt").write_text(REPO_COMMIT + "\n")
print("Pinned repository and runtime patches are ready.")

## 4. Hugging Face authentication and token check

Paste your token into the widget. It is never written to the notebook or the result files.

In [ ]:
from huggingface_hub import notebook_login
from transformers import AutoTokenizer

notebook_login()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="right")
true_ids = tokenizer("True", add_special_tokens=False).input_ids
false_ids = tokenizer("False", add_special_tokens=False).input_ids
assert len(true_ids) == len(false_ids) == 1, (true_ids, false_ids)
print("True token:", true_ids, "False token:", false_ids)

## 5. Shared helpers

* `run_and_snapshot` streams a command, logs it, and copies every file it changed in `results/` into its own run directory, so no split can overwrite another.
* `in_repo()` scopes the working directory: the released code resolves `data/`, `results/` and `saved_models/` relative to the repository root.
* `labelled_turns` replays the exact iteration order of `LMKTDatasetPacked` at test time and attaches the real turn id, student text and reference answer. The export in Section 7 asserts this alignment row by row.

In [ ]:
import re, time, contextlib
from types import SimpleNamespace
from fractions import Fraction
import html
import pandas as pd
from sklearn.metrics import roc_auc_score

RESULTS = REPO / "results"
RESULTS.mkdir(exist_ok=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

@contextlib.contextmanager
def in_repo():
    previous = Path.cwd()
    os.chdir(REPO)
    try:
        yield
    finally:
        os.chdir(previous)

def changed_result_files(before):
    return [p for p in RESULTS.glob("*") if p.is_file() and before.get(p) != p.stat().st_mtime_ns]

def run_and_snapshot(cmd, target, seed):
    target.mkdir(parents=True, exist_ok=True)
    before = {p: p.stat().st_mtime_ns for p in RESULTS.glob("*") if p.is_file()}
    print("COMMAND:", " ".join(cmd))
    env = os.environ.copy()
    env["DKT_SEED"] = str(seed)
    started = time.time()
    with (target / "command.log").open("w") as log:
        process = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return_code = process.wait()
    elapsed = time.time() - started
    (target / "elapsed_seconds.txt").write_text(f"{elapsed:.3f}\n")
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
    copied = []
    for src in changed_result_files(before):
        shutil.copy2(src, target / src.name)
        copied.append(src.name)
    (target / "artifacts.json").write_text(json.dumps(copied, indent=2))
    print(f"Saved {len(copied)} result files to {target}; elapsed {elapsed/60:.1f} min")
    return target

def lmkt_command(action, dataset, model_name=None):
    cmd = [sys.executable, "-u", "-m", "dialogue_kt.main", action,
           "--dataset", dataset, "--model_type", "lmkt", "--base_model", BASE_MODEL,
           "--agg", "mean-ar", "--pack_kcs", "true", "--quantize", "true"]
    if model_name:
        cmd += ["--model_name", model_name]
    if action == "train":
        cmd += ["--epochs", "1" if SMOKE else "2", "--lr", "2e-4", "--r", "8",
                "--lora_alpha", "16", "--batch_size", "1", "--grad_accum_steps", "16"]
    if SMOKE:
        cmd.append("--debug")
    return cmd

def repo_args(dataset):
    return SimpleNamespace(dataset=dataset, split_by_subject=False, typical_cutoff=1, tag_src="atc",
                           prompt_inc_labels=False, pack_kcs=True, agg="mean-ar", quantize=True,
                           inc_first_label=False, debug=False, model_type="lmkt")

def adapter_names(dataset, seed):
    # Mirrors training.py: the checkpoint is saved as model_name + f"_{fold}" when a default fold exists.
    from dialogue_kt.data_loading import get_default_fold
    fold = get_default_fold(repo_args(dataset))
    model_name = f"qlora_{dataset}_s{seed}_{RUN_ID}"
    return model_name, model_name + (f"_{fold}" if fold else "")

def load_split_frame(dataset, split):
    from dialogue_kt.data_loading import load_annotated_data, get_default_fold
    args = repo_args(dataset)
    with in_repo():
        _, val_df, test_df = load_annotated_data(args, get_default_fold(args))
    frame = {"val": val_df, "test": test_df}[split]
    return frame.head(SMOKE_DIALOGUES) if SMOKE else frame

NUMBER = r"[-+]?\d[\d,]*(?:\.\d+)?(?:/\d+(?:\.\d+)?)?"

def final_number(text):
    lines = [line.strip() for line in html.unescape(str(text)).splitlines() if line.strip()]
    if not lines:
        return None
    matches = re.findall(NUMBER, lines[-1])
    return matches[-1].replace(",", "") if matches else None

def reference_answer(dataset, sample):
    if dataset == "mathdial":
        return final_number(sample["meta_data"]["correct_solution"])
    return None   # CoMTA releases an outcome label ("Answer Accepted"), not a reference answer.

def labelled_turns(dataset, frame, skip_first_turn=True, with_context=False):
    from dialogue_kt.kt_data_loading import apply_annotations
    rows = []
    for idx, sample in frame.iterrows():
        dialogue = apply_annotations(sample)
        if not dialogue:
            continue
        first_turn, history = True, []
        for turn in dialogue:
            teacher = str(turn.get("teacher") or "").strip()
            if turn["correct"] is not None:
                if skip_first_turn and first_turn:   # LMKTDatasetPacked(skip_first_turn=True) at test time
                    first_turn = False
                else:
                    first_turn = False
                    row = {"dialogue": str(idx), "turn_id": int(turn["turn"]), "label": int(turn["correct"]),
                           "kcs": list(turn["kcs"]), "student_text": str(turn["student"]),
                           "reference": reference_answer(dataset, sample)}
                    if with_context:
                        row["history"] = "\n".join(history + ([f"Tutor: {teacher}"] if teacher else []))[-2500:]
                        row["problem"] = sample["meta_data"].get("question") if dataset == "mathdial" else None
                    rows.append(row)
            if teacher:
                history.append(f"Tutor: {teacher}")
            history.append(f"Student: {turn['student']}")
    last = {row["dialogue"]: i for i, row in enumerate(rows)}
    for i, row in enumerate(rows):
        row["final_turn"] = last[row["dialogue"]] == i
    return rows

def rows_path(dataset, model_key, seed, split):
    return run_dir(dataset, model_key, seed, split) / "raw_logit_gaps.json"

def load_rows(dataset, model_key, seed, split):
    rows = json.loads(rows_path(dataset, model_key, seed, split).read_text())
    for row in rows:
        row["gaps"] = np.asarray(row["logit_gaps"], dtype=float)
    return rows

def mean_prediction(rows):
    return np.array([np.mean(1 / (1 + np.exp(-row["gaps"]))) for row in rows])

def safe_auc(y, p):
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan")

def parse_official_metrics(directory):
    files = sorted(directory.glob("metrics_*.txt"))
    if len(files) != 1:
        raise FileNotFoundError(f"Expected one metrics file in {directory}; found {files}")
    text = files[0].read_text()
    value = r"([0-9.]+|nan)"
    out = {"file": files[0].name}
    loss = re.search(r"Loss:\s*" + value, text)
    blocks = {"": r"Overall \((\d+) samples\):", "final_": r"Final Turn \((\d+) samples\):"}
    for prefix, header in blocks.items():
        match = re.search(header + r".*?Acc: " + value + ", AUC: " + value + ", Prec: " + value +
                          ", Rec: " + value + ", F1: " + value, text, re.S)
        if match:
            out[prefix + "n"] = int(match.group(1))
            for key, val in zip(["accuracy", "auc", "precision", "recall", "f1"], match.groups()[1:]):
                out[prefix + key] = float(val)
    if loss:
        out["loss"] = float(loss.group(1))
    return out

print("Helpers ready.")

## 5b. Private artifact store

Training the MathDial adapter can take hours, so trained adapters are saved **privately** as soon as each training run finishes, and the final result ZIP is saved in Section 17.

* `ARTIFACT_STORE = "drive"` — your Google Drive, `MyDrive/soict2026_artifacts/{full|smoke}/`. Colab asks for Drive permission when this cell runs.
* `ARTIFACT_STORE = "hub"` — a Hugging Face repository `HF_ARTIFACT_REPO`, created private; the cell refuses to write to a public repository. The Hugging Face token from Section 4 needs **write** permission.
* `ARTIFACT_STORE = "none"` — nothing is saved outside the runtime.

`adapters_index.json` records run ID, dataset, seed, base model, repository commit, library versions and test AUC for every adapter. Next time set `ADAPTER_SOURCE = "store"` (optionally `REUSE_RUN_ID`): Section 6 downloads the matching adapter and runs only the released test pass instead of training.

Keep the store private. The CoMTA license permits internal, non-commercial evaluation only and prohibits training on or redistributing the dataset or its derivatives; exported CoMTA rows must stay private, and with `COMTA_MODE = "evaluate_only"` no CoMTA adapter is created.

In [ ]:
def store_location():
    if ARTIFACT_STORE == "drive":
        return f"Google Drive {DRIVE_ROOT / STORE_PREFIX}"
    return f"private Hugging Face repo {HF_ARTIFACT_REPO} ({STORE_PREFIX}/)"

def store_ready():
    if ARTIFACT_STORE == "drive":
        if not Path("/content/drive/MyDrive").exists():
            from google.colab import drive
            drive.mount("/content/drive")
        (DRIVE_ROOT / STORE_PREFIX).mkdir(parents=True, exist_ok=True)
    elif ARTIFACT_STORE == "hub":
        from huggingface_hub import HfApi, create_repo
        assert HF_ARTIFACT_REPO, "Set HF_ARTIFACT_REPO, e.g. 'your-hf-name/soict2026-private-artifacts'."
        create_repo(HF_ARTIFACT_REPO, private=True, exist_ok=True)
        assert HfApi().repo_info(HF_ARTIFACT_REPO).private, f"{HF_ARTIFACT_REPO} is public; use a private repository."
    print("Artifact store:", store_location())

def store_read_index():
    if ARTIFACT_STORE == "drive":
        path = DRIVE_ROOT / STORE_PREFIX / "adapters_index.json"
        return json.loads(path.read_text()) if path.exists() else []
    from huggingface_hub import HfApi, hf_hub_download
    name = f"{STORE_PREFIX}/adapters_index.json"
    if not HfApi().file_exists(HF_ARTIFACT_REPO, name):
        return []
    return json.loads(Path(hf_hub_download(HF_ARTIFACT_REPO, name, force_download=True)).read_text())

def store_write_index(index):
    payload = json.dumps(index, indent=2)
    if ARTIFACT_STORE == "drive":
        (DRIVE_ROOT / STORE_PREFIX / "adapters_index.json").write_text(payload)
    else:
        from huggingface_hub import HfApi
        HfApi().upload_file(path_or_fileobj=payload.encode(), path_in_repo=f"{STORE_PREFIX}/adapters_index.json",
                            repo_id=HF_ARTIFACT_REPO, commit_message=f"Update adapter index ({RUN_ID})")

def store_save_adapter(dataset, seed, model_name, saved_name):
    local = REPO / "saved_models" / saved_name
    official = parse_official_metrics(run_dir(dataset, "qlora", seed, "test"))
    entry = {"run_id": RUN_ID, "dataset": dataset, "seed": seed, "model_name": model_name, "saved_name": saved_name,
             "smoke": SMOKE, "base_model": BASE_MODEL, "repository_commit": REPO_COMMIT,
             "created_utc": datetime.now(timezone.utc).isoformat(), "versions": VERSIONS,
             "test_n": official.get("n"), "test_auc": official.get("auc")}
    if ARTIFACT_STORE == "drive":
        shutil.copytree(local, DRIVE_ROOT / STORE_PREFIX / "adapters" / saved_name, dirs_exist_ok=True)
    else:
        from huggingface_hub import HfApi
        HfApi().upload_folder(folder_path=str(local), path_in_repo=f"{STORE_PREFIX}/adapters/{saved_name}",
                              repo_id=HF_ARTIFACT_REPO, commit_message=f"Adapter {saved_name}")
    store_write_index([e for e in store_read_index() if e["saved_name"] != saved_name] + [entry])
    print(f"Saved adapter {saved_name} to {store_location()}")

def legacy_drive_entries():
    # Manual backups made before the index existed: DRIVE_ROOT/<RUN_ID>/adapters.json + adapters/<saved_name>/
    from dialogue_kt.data_loading import get_default_fold
    entries = []
    for mapping in sorted(DRIVE_ROOT.glob("*/adapters.json")):
        run_id = mapping.parent.name
        if run_id.endswith("_smoke") != SMOKE:
            continue
        for key, value in json.loads(mapping.read_text()).items():
            saved_name = value["adapter"] if isinstance(value, dict) else value
            dataset, seed = key.split("/seed")
            fold = get_default_fold(repo_args(dataset))
            suffix = f"_{fold}" if fold else ""
            model_name = saved_name[: -len(suffix)] if suffix and saved_name.endswith(suffix) else saved_name
            entries.append({"run_id": run_id, "dataset": dataset, "seed": int(seed), "model_name": model_name,
                            "saved_name": saved_name, "smoke": SMOKE, "base_model": BASE_MODEL,
                            "repository_commit": REPO_COMMIT, "created_utc": run_id, "test_auc": None,
                            "source_dir": str(mapping.parent / "adapters" / saved_name)})
    return entries

def store_fetch_adapter(dataset, seed):
    index = store_read_index()
    if ARTIFACT_STORE == "drive":
        index += [e for e in legacy_drive_entries() if e["saved_name"] not in {i["saved_name"] for i in index}]
    candidates = [e for e in index
                  if e["dataset"] == dataset and e["seed"] == seed and e["base_model"] == BASE_MODEL
                  and e["repository_commit"] == REPO_COMMIT and (REUSE_RUN_ID is None or e["run_id"] == REUSE_RUN_ID)]
    assert candidates, f"No stored adapter for {dataset} seed {seed} in {store_location()} (REUSE_RUN_ID={REUSE_RUN_ID})."
    entry = max(candidates, key=lambda e: e["created_utc"])
    local = REPO / "saved_models" / entry["saved_name"]
    if ARTIFACT_STORE == "drive":
        source = Path(entry.get("source_dir") or DRIVE_ROOT / STORE_PREFIX / "adapters" / entry["saved_name"])
        shutil.copytree(source, local, dirs_exist_ok=True)
    else:
        from huggingface_hub import snapshot_download
        prefix = f"{STORE_PREFIX}/adapters/{entry['saved_name']}"
        downloaded = Path(snapshot_download(HF_ARTIFACT_REPO, allow_patterns=[f"{prefix}/*"], local_dir=str(WORK / "hub_download")))
        shutil.copytree(downloaded / prefix, local, dirs_exist_ok=True)
    assert (local / "adapter_config.json").exists(), f"Incomplete adapter at {local}"
    print(f"Reusing adapter {entry['saved_name']} trained in run {entry['run_id']} (test AUC {entry['test_auc']})")
    return entry

def store_save_results(archive):
    archive = Path(archive)
    if ARTIFACT_STORE == "drive":
        target = DRIVE_ROOT / STORE_PREFIX / "results"
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive, target / archive.name)
    elif ARTIFACT_STORE == "hub":
        from huggingface_hub import HfApi
        HfApi().upload_file(path_or_fileobj=str(archive), path_in_repo=f"{STORE_PREFIX}/results/{archive.name}",
                            repo_id=HF_ARTIFACT_REPO, commit_message=f"Result bundle {RUN_ID}")
    print("Result bundle saved to", store_location())

if ARTIFACT_STORE != "none":
    store_ready()
    print("Stored adapters:", [(e["dataset"], e["seed"], e["run_id"]) for e in store_read_index()])

## 6. REAL DATA — matched zero-shot and QLoRA runs

Zero-shot inference is seed-independent and runs once per dataset. QLoRA trains once per seed on MathDial; training evaluates the selected checkpoint on test at the end. With `COMTA_MODE = "evaluate_only"`, CoMTA is never trained on: the MathDial adapter is exposed under the name the released test loop expects (a symbolic link, no copy) and evaluated on the CoMTA test split. Validation predictions are produced by the raw-logit export in Section 7, which avoids a second subprocess pass per model.

In [ ]:
def cross_evaluation_names(dataset, source_saved_name):
    # The released test loop appends the dataset's default fold to --model_name, so the source adapter is
    # exposed under that name through a symbolic link instead of being copied or retrained.
    from dialogue_kt.data_loading import get_default_fold
    fold = get_default_fold(repo_args(dataset))
    model_name = f"{source_saved_name}_on_{dataset}"
    return model_name, model_name + (f"_{fold}" if fold else "")

ADAPTERS, ADAPTER_PROVENANCE = {}, {}
for dataset in FIT_ORDER:
    print(f"\n===== {dataset.upper()} ({'trains its own adapter' if FIT_ON[dataset] == dataset else 'evaluation only, adapter from ' + FIT_ON[dataset]}) =====")
    if RUN_ZERO_SHOT:
        run_and_snapshot(lmkt_command("test", dataset), run_dir(dataset, "zero_shot", None, "test"), PRIMARY_SEED)
    if RUN_QLORA:
        for seed in SEEDS:
            target = run_dir(dataset, "qlora", seed, "test")
            source = FIT_ON[dataset]
            if source != dataset:
                source_saved = ADAPTERS[(source, seed)]
                model_name, saved_name = cross_evaluation_names(dataset, source_saved)
                link = REPO / "saved_models" / saved_name
                if not link.exists():
                    link.symlink_to((REPO / "saved_models" / source_saved).resolve(), target_is_directory=True)
                run_and_snapshot(lmkt_command("test", dataset, model_name=model_name), target, seed)
                provenance = {**ADAPTER_PROVENANCE[f"{source}/seed{seed}"], "trained_on": source, "evaluated_on": dataset}
                provenance.pop("adapter", None)
            elif ADAPTER_SOURCE == "store":
                entry = store_fetch_adapter(dataset, seed)
                run_and_snapshot(lmkt_command("test", dataset, model_name=entry["model_name"]), target, seed)
                saved_name = entry["saved_name"]
                provenance = {"source": "store", "trained_in_run": entry["run_id"], "trained_on": dataset}
            else:
                model_name, saved_name = adapter_names(dataset, seed)
                run_and_snapshot(lmkt_command("train", dataset, model_name=model_name), target, seed)
                provenance = {"source": "trained", "trained_in_run": RUN_ID, "trained_on": dataset}
            assert (REPO / "saved_models" / saved_name).exists(), f"Adapter not found: {saved_name}"
            if ADAPTER_SOURCE == "train" and ARTIFACT_STORE != "none" and source == dataset:
                store_save_adapter(dataset, seed, model_name, saved_name)
            ADAPTERS[(dataset, seed)] = saved_name
            ADAPTER_PROVENANCE[f"{dataset}/seed{seed}"] = {"adapter": saved_name, **provenance}

(RUN_ROOT / "adapters.json").write_text(json.dumps(ADAPTER_PROVENANCE, indent=2))

## 7. REAL DATA — raw per-KC logit export and golden test

One base model is loaded per dataset. QLoRA seeds are attached as named adapters and zero-shot inference runs inside `disable_adapter()`, so each dataset costs a single model load. Every row carries `dialogue, turn_id, label, kcs, logit_gaps, student_text, reference, final_turn`.

**Golden test.** For the test split, the mean of `sigmoid(logit_gaps)` must reproduce the released implementation's `mean-ar` prediction: sample counts must match exactly and AUC within `GOLDEN_AUC_TOL` points. This verifies the hand-built attention mask against the released loop under the installed library versions.

In [ ]:
import gc
from contextlib import nullcontext

def export_split(model, tok, dataset, split, model_key, seed):
    from dialogue_kt.kt_data_loading import LMKTDatasetPacked, LMKTCollatorPacked, get_dataloader
    from dialogue_kt.prompting import get_true_false_tokens
    frame = load_split_frame(dataset, split)
    with in_repo():
        meta = labelled_turns(dataset, frame)
        data = LMKTDatasetPacked(frame, tok, repo_args(dataset), skip_first_turn=True)
    assert len(data.data) == len(meta), (len(data.data), len(meta))
    for sample, row in zip(data.data, meta):
        assert str(sample["dialogue_idx"]) == row["dialogue"]
        assert int(sample["label"]) == row["label"] and list(sample["kcs"]) == row["kcs"]
    loader = get_dataloader(data, LMKTCollatorPacked(tok), EXPORT_BATCH_SIZE, False)
    true_token, false_token = get_true_false_tokens(tok)
    gaps = []
    with torch.inference_mode():
        for batch in loader:
            mask = batch["attention_mask"].clone()
            mask[mask == 0] = torch.finfo(model.dtype).min
            mask[mask == 1] = 0
            mask = mask.type(model.dtype)
            output = model(input_ids=batch["input_ids"], attention_mask=mask, position_ids=batch["position_ids"])
            for b in range(output.logits.shape[0]):
                n_kcs = int(batch["num_kcs"][b])
                positions = batch["last_idxs"][b, :n_kcs].to(output.logits.device)
                token_logits = output.logits[b, positions].float()
                gaps.append((token_logits[:, true_token] - token_logits[:, false_token]).cpu().numpy().tolist())
    assert len(gaps) == len(meta)
    rows = [dict(row, logit_gaps=g) for row, g in zip(meta, gaps)]
    path = rows_path(dataset, model_key, seed, split)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(rows))
    print(f"{dataset} {model_key} {seed_tag(model_key, seed)} {split}: {len(rows)} rows -> {path}")
    return rows

def golden_check(dataset, model_key, seed):
    rows = load_rows(dataset, model_key, seed, "test")
    official = parse_official_metrics(run_dir(dataset, model_key, seed, "test"))
    y = np.array([r["label"] for r in rows])
    auc_export = safe_auc(y, mean_prediction(rows)) * 100
    final_n = sum(r["final_turn"] for r in rows)
    record = {"dataset": dataset, "model": model_key, "seed": seed_tag(model_key, seed),
              "n_export": len(rows), "n_official": official.get("n"),
              "final_n_export": final_n, "final_n_official": official.get("final_n"),
              "auc_export": auc_export, "auc_official": official.get("auc")}
    assert record["n_export"] == record["n_official"], record
    assert record["final_n_export"] == record["final_n_official"], record
    if not (np.isnan(auc_export) or np.isnan(record["auc_official"])):
        assert abs(auc_export - record["auc_official"]) <= GOLDEN_AUC_TOL, record
    return record

from dialogue_kt.models.lm import get_model
GOLDEN = []
for dataset in DATASETS:
    first_adapter = ADAPTERS.get((dataset, SEEDS[0])) if RUN_QLORA else None
    with in_repo():
        model, tok = get_model(BASE_MODEL, True, model_name=first_adapter, quantize=True)
        if RUN_QLORA:
            for seed in SEEDS[1:]:
                model.load_adapter(f"saved_models/{ADAPTERS[(dataset, seed)]}", adapter_name=f"seed{seed}")
    model.eval()
    for model_key, seed in MODEL_RUNS:
        if model_key == "zero_shot":
            context = model.disable_adapter() if RUN_QLORA else nullcontext()
        else:
            model.set_adapter("default" if seed == SEEDS[0] else f"seed{seed}")
            context = nullcontext()
        with context:
            for split in ["val", "test"]:
                export_split(model, tok, dataset, split, model_key, seed)
        GOLDEN.append(golden_check(dataset, model_key, seed))
    del model
    gc.collect()
    torch.cuda.empty_cache()

golden_df = pd.DataFrame(GOLDEN)
golden_df.to_csv(RUN_ROOT / "golden_test.csv", index=False)
display(golden_df)

## 8. REAL DATA — metrics computed from predictions

All metrics are recomputed with scikit-learn from the exported predictions (`mean-ar` of per-KC probabilities, threshold 0.5 exactly as `np.round` in the released code). The released `metrics_*.txt` values are kept alongside as `official_*` for cross-checking. Comparisons are valid only within the same dataset and sample count.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def threshold_metrics(y, p, threshold=None):
    y = np.asarray(y)
    hard = np.round(p) if threshold is None else (p >= threshold).astype(float)
    precision, recall, f1, _ = precision_recall_fscore_support(y, hard, average="binary", zero_division=0)
    return {"n": len(y), "accuracy": 100 * accuracy_score(y, hard), "auc": 100 * safe_auc(y, p),
            "precision": 100 * precision, "recall": 100 * recall, "f1": 100 * f1}

metric_rows = []
for dataset in DATASETS:
    for model_key, seed in MODEL_RUNS:
        for split in ["val", "test"]:
            rows = load_rows(dataset, model_key, seed, split)
            y = np.array([r["label"] for r in rows])
            p = mean_prediction(rows)
            final = np.array([r["final_turn"] for r in rows])
            record = {"dataset": dataset, "model": model_key, "seed": seed_tag(model_key, seed), "split": split}
            record.update(threshold_metrics(y, p))
            record.update({"final_" + k: v for k, v in threshold_metrics(y[final], p[final]).items()})
            if split == "test":
                official = parse_official_metrics(run_dir(dataset, model_key, seed, "test"))
                record.update({"official_" + k: v for k, v in official.items() if k != "file"})
            metric_rows.append(record)

metrics_df = pd.DataFrame(metric_rows)
assert not metrics_df.empty, "No exported predictions found; run Sections 6-7 first."
metrics_df = metrics_df.sort_values(["dataset", "split", "model", "seed"])
metrics_df.to_csv(RUN_ROOT / "metrics_summary.csv", index=False)
display(metrics_df)

## 9. REAL DATA — validation-selected threshold and confusion matrices

The released implementation fixes the threshold at 0.5. Here a threshold is selected by validation F1 only, frozen, and applied to the untouched test split.

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

threshold_rows = []
for dataset in FIT_ORDER:
    for model_key, seed in MODEL_RUNS:
        val = load_rows(FIT_ON[dataset], model_key, seed, "val")   # CoMTA (evaluation only) reuses the MathDial threshold
        test = load_rows(dataset, model_key, seed, "test")
        y_val, p_val = np.array([r["label"] for r in val]), mean_prediction(val)
        y_test, p_test = np.array([r["label"] for r in test]), mean_prediction(test)
        candidates = np.linspace(0.05, 0.95, 181)
        scores = np.array([f1_score(y_val, p_val >= t, zero_division=0) for t in candidates])
        tied = candidates[scores == scores.max()]
        threshold = float(tied[np.argmin(np.abs(tied - 0.5))])
        record = {"dataset": dataset, "model": model_key, "seed": seed_tag(model_key, seed), "threshold": threshold,
                  "threshold_fit_on": FIT_ON[dataset]}
        record.update(threshold_metrics(y_test, p_test, threshold))
        threshold_rows.append(record)
        cm = confusion_matrix(y_test, (p_test >= threshold).astype(int), labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=["Incorrect", "Correct"]).plot(cmap="Blues", colorbar=False)
        plt.title(f"{dataset} / {model_key} / {seed_tag(model_key, seed)} — threshold={threshold:.3f}")
        plt.tight_layout()
        plt.savefig(RUN_ROOT / f"confusion_{dataset}_{model_key}_{seed_tag(model_key, seed)}.png", dpi=180)
        plt.show()

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(RUN_ROOT / "threshold_calibrated_metrics.csv", index=False)
display(threshold_df)

## 10. SAVA — verify-or-abstain cascade

The verifier used for the manuscript's real-turn numbers, vendored with one addition: every verdict reports the **stage** that produced it, so coverage can be attributed layer by layer.

| Stage | Outcome |
|---|---|
| `question_guard` | abstain: a trailing `?` makes the turn non-answer-bearing |
| extraction: `direct` → `unit_suffix` → `answer_span` | locate a candidate expression |
| extraction: `conclusion_cue` | when no expression is found, the last number after the last conclusion cue ("so", "the answer is", ...) is verified; verdicts are reported as `conclusion_cas_equivalent` / `conclusion_cas_not_equivalent` |
| `prose_guard` | abstain: no answer-bearing expression |
| `parse_guard` | abstain: unparseable or a product of prose letters |
| `symbolic_residue_guard` | abstain: free symbols against a numeric reference |
| `cas_equivalent` | **correct** |
| `approx_constant_guard`, `rounded_decimal_guard` | abstain: a reporting choice, not an error |
| `cas_not_equivalent` | **incorrect** |

In [ ]:
import sympy as sp
from sympy import simplify
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor)

SAVA_TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)
_FILLER = r"""\b(the|answer|is|are|it|i|think|we|get|got|would|be|equals?|equal|to|
              so|then|therefore|thus|about|approximately|approx|roughly|maybe|
              apples|dollars|cents|cm|mm|m|km|kg|g|inches|inch|ft|feet|units?|
              degrees|hours|minutes|seconds|students|books|pens|marbles)\b"""
_NUM_UNIT = re.compile(r"^\s*(-?\d+\.?\d*(?:/\d+\.?\d*)?)\s*([a-z]+(?:\s+[a-z]+)?)\s*$")
_CONCL_CUE = re.compile(r"\b(so|therefore|thus|hence|in total|altogether|the answer|"
                        r"answer is|total is|which is|that (?:is|means)|equals)\b")
_QUESTION_TAIL = re.compile(r"\?\s*$")
_APPROX_CONSTS = ("pi", "E", "sqrt", "exp", "log")

def _extract(text):
    t = str(text).strip().lower()
    t = re.sub(r"n't\b", " not ", t)
    t = re.sub(r"'(s|re|ll|ve|m|d)\b", " ", t)
    t = t.replace("'", " ").replace("$", "").replace(",", "")
    if "=" in t:
        t = t.split("=")[-1]
    t = re.sub(_FILLER, " ", t, flags=re.VERBOSE)
    t = t.replace("%", "/100")
    t = re.sub(r"[^0-9a-z+\-*/^().\s]", " ", t).strip()
    return t or None

def _parse(s):
    if not s:
        return None
    try:
        return parse_expr(s, transformations=SAVA_TRANSFORMS, evaluate=True)
    except Exception:
        m = re.search(r"-?\d+\.?\d*(/\d+\.?\d*)?", s)
        if m:
            try:
                return parse_expr(m.group(0), transformations=SAVA_TRANSFORMS)
            except Exception:
                return None
        return None

def _equivalent(a, b, rtol=1e-9):
    try:
        if simplify(a - b) == 0:
            return True
    except Exception:
        pass
    try:
        fa, fb = float(a.evalf()), float(b.evalf())
        return abs(fa - fb) <= rtol * max(1.0, abs(fb))
    except Exception:
        return False

def _looks_mathematical(s):
    if not s:
        return False
    if not re.search(r"\d", s) and not re.search(r"[+\-*/^]", s):
        return False
    if len(set(re.findall(r"[a-z]", s))) > 2:
        return False
    return not re.search(r"[a-z]{3,}", s)

def _is_word_product(expr):
    try:
        syms = expr.free_symbols
        return len(syms) > 2 and all(len(str(x)) == 1 for x in syms)
    except Exception:
        return False

def _is_symbolic_constant(s):
    return any(c in str(s) for c in _APPROX_CONSTS)

def _is_rounded_decimal_of(sa, ra):
    try:
        s_str = str(sa)
        if "." not in s_str:
            return False
        dps = len(s_str.split(".")[1].rstrip("0"))
        if dps == 0 or dps > 8:
            return False
        rf = float(ra.evalf())
        if abs(rf - round(rf)) < 1e-12:
            return False
        return abs(float(sa) - round(rf, dps)) <= 0.5 * 10 ** (-dps)
    except Exception:
        return False

def _strip_unit_suffix(text):
    t = str(text).strip().lower()
    if len(t) > 24:
        return None
    m = _NUM_UNIT.match(t)
    return m.group(1) if m else None

def _sentences(text):
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n+", str(text)) if p.strip()]

def _locate_answer_span(text):
    t = str(text).strip()
    if _QUESTION_TAIL.search(t):
        return None
    sents = _sentences(t)
    if len(sents) <= 1:
        return None
    numbered = [s for s in sents if re.search(r"\d", s)]
    if not numbered:
        return None
    last_num = numbered[-1]
    idx = {id(s): i for i, s in enumerate(sents)}
    cued = [s for s in numbered if _CONCL_CUE.search(s.lower())]
    if cued and idx[id(cued[-1])] > idx[id(last_num)]:
        return cued[-1]
    return last_num

def _last_number_expr(span):
    m = re.findall(r"-?\d+\.?\d*\s*/\s*\d+\.?\d*|-?\d+\.?\d*", str(span).replace(",", ""))
    return m[-1].replace(" ", "") if m else None

def _conclusion_number(text):
    """Last number after the last conclusion cue ("so", "the answer is", ...), for one-clause conclusions."""
    t = str(text).replace(",", "")
    cues = list(_CONCL_CUE.finditer(t.lower()))
    if not cues:
        return None
    nums = re.findall(r"-?\d+\.?\d*\s*/\s*\d+\.?\d*|-?\d+\.?\d*", t[cues[-1].end():])
    return nums[-1].replace(" ", "") if nums else None

def sava_verify(student_answer, reference_answer, approx_tol=5e-3, question_guard_global=None):
    guard = SAVA_QUESTION_GUARD_GLOBAL if question_guard_global is None else question_guard_global
    def verdict(label, stage, extracted, reference, extraction):
        y = {"correct": 1, "incorrect": 0}.get(label)
        return {"verdict": label, "y": y, "stage": stage, "extraction": extraction,
                "extracted": extracted, "reference": reference}
    if reference_answer is None:
        return verdict("undetermined", "no_reference", None, None, None)
    if guard and _QUESTION_TAIL.search(str(student_answer).strip()):
        return verdict("undetermined", "question_guard", None, _extract(reference_answer), None)
    se, re_ = _extract(student_answer), _extract(reference_answer)
    extraction = "direct"
    if not _looks_mathematical(se):
        unit_num = _strip_unit_suffix(student_answer)
        if unit_num is not None:
            se, extraction = unit_num, "unit_suffix"
    if not _looks_mathematical(se):
        span = _locate_answer_span(student_answer)
        if span is not None:
            cand = _last_number_expr(span)
            if cand is not None:
                se, extraction = cand, "answer_span"
    if not _looks_mathematical(se):
        concl = _conclusion_number(student_answer)
        if concl is not None:
            alt = sava_verify(concl, reference_answer, approx_tol, guard)
            if alt["verdict"] != "undetermined":
                return dict(alt, stage="conclusion_" + alt["stage"], extraction="conclusion_cue")
        return verdict("undetermined", "prose_guard", se, re_, extraction)
    sa, ra = _parse(se), _parse(re_)
    if sa is None or ra is None or _is_word_product(sa):
        return verdict("undetermined", "parse_guard", se, re_, extraction)
    if not ra.free_symbols and sa.free_symbols:
        return verdict("undetermined", "symbolic_residue_guard", str(sa), str(ra), extraction)
    if _equivalent(sa, ra):
        return verdict("correct", "cas_equivalent", str(sa), str(ra), extraction)
    if _is_symbolic_constant(re_ or "") or _is_symbolic_constant(reference_answer):
        try:
            if abs(float(sa.evalf()) - float(ra.evalf())) <= approx_tol * max(1.0, abs(float(ra.evalf()))):
                return verdict("undetermined", "approx_constant_guard", str(sa), str(ra), extraction)
        except Exception:
            pass
    if _is_rounded_decimal_of(sa, ra):
        return verdict("undetermined", "rounded_decimal_guard", str(sa), str(ra), extraction)
    return verdict("incorrect", "cas_not_equivalent", str(sa), str(ra), extraction)

SAVA_UNIT_TESTS = [
    ("3/6", "1/2", "correct"), ("2*7", "14", "correct"), ("27 miles", "27", "correct"),
    ("15", "14", "incorrect"), ("is it 14?", "14", "undetermined"), ("2x + 3", "14", "undetermined"),
    ("3.14", "pi", "undetermined"),
    ("First I subtract 3 from both sides and get 2x = 8. Then I divide by 2. So x = 4.", "4", "correct"),
    ("I think x = 8/2", "4", "correct"), ("$1,200", "1200", "correct"),
    ("I would move the constant to the other side first.", "4", "undetermined"),
    ("4.33", "13/3", "undetermined"), ("2*(x+1)", "2x + 2", "correct"),
    ("First I added them up, so the answer is 12.", "12", "correct"),
    ("First I added them up, so the answer is 11.", "12", "incorrect"),
]
for student, reference, expected in SAVA_UNIT_TESTS:
    got = sava_verify(student, reference, question_guard_global=True)
    assert got["verdict"] == expected, (student, reference, expected, got)
print(f"SAVA unit tests passed ({len(SAVA_UNIT_TESTS)} cases).")

## 11. SAVA — CONSTRUCTED BENCHMARK and REAL-TURN AGREEMENT

**Constructed benchmark.** For each of the first `SAVA_N_REFERENCES` MathDial test references with a usable intermediate step and a distinct learner final answer, twelve candidates are built as plain strings with Python `Fraction` arithmetic (the generator never calls SymPy):

* 6 correct: identity, decimal padding, unit suffix, conclusion sentence, unreduced fraction `2x/2`, split sum `(x-1) + 1`;
* 6 incorrect: off-by-one, sign flip, ×10 scale, wrong conclusion sentence, an **intermediate number from the real reference solution**, and the **real learner's incorrect final answer**.

Classes are balanced 1:1. Baselines: `always_correct`, `exact_string`, and a last-number `numeric_regex`. Abstention counts as an error in `accuracy_all` and `balanced_accuracy`.

**Real-turn agreement.** On MathDial test turns, SAVA's verdicts are compared with the released turn-level correctness labels. These labels judge each turn, while SAVA compares against the final reference answer; disagreement on intermediate turns is therefore expected and is reported separately for final and non-final turns.

In [ ]:
from scipy.stats import binomtest

def fmt(q):
    q = Fraction(q)
    return str(q.numerator) if q.denominator == 1 else format(float(q), ".12g")

def to_fraction(text):
    try:
        return Fraction(str(text).replace(",", ""))
    except (ValueError, ZeroDivisionError):
        return None

def numeric_regex(reference, candidate):
    matches = re.findall(NUMBER, str(candidate))
    if not matches:
        return "undetermined"
    value = to_fraction(matches[-1])
    if value is None:
        return "undetermined"
    return "correct" if value == to_fraction(reference) else "incorrect"

def exact_string(reference, candidate):
    return "correct" if str(candidate).strip().lower() == fmt(to_fraction(reference)) else "incorrect"

def make_candidates(q, intermediate, learner_final):
    return [
        ("identity", fmt(q), "correct"),
        ("decimal_padding", (fmt(q) + ".0") if q.denominator == 1 else fmt(q) + "0", "correct"),
        ("unit_suffix", f"{fmt(q)} units", "correct"),
        ("conclusion_sentence", f"First I added them up, so the answer is {fmt(q)}.", "correct"),
        ("unreduced_fraction", f"{fmt(2 * q)}/2", "correct"),
        ("split_sum", f"{fmt(q - 1)} + 1", "correct"),
        ("off_by_one", fmt(q + 1), "incorrect"),
        ("sign_flip", fmt(-q), "incorrect"),
        ("scale_x10", fmt(10 * q), "incorrect"),
        ("wrong_conclusion_sentence", f"First I added them up, so the answer is {fmt(q - 1)}.", "incorrect"),
        ("intermediate_step", fmt(intermediate), "incorrect"),
        ("learner_incorrect_final", fmt(learner_final), "incorrect"),
    ]

sava_benchmark_df = pd.DataFrame()
sava_summary_df = pd.DataFrame()
sava_real_df = pd.DataFrame()
if RUN_SAVA and "mathdial" in DATASETS:
    source = pd.read_csv(REPO / "data/annotated/mathdial_test_atc.csv")
    records, n_refs = [], 0
    for _, item in source.iterrows():
        q = to_fraction(final_number(item["ground_truth"]))
        learner = to_fraction(final_number(item["student_incorrect_solution"]))
        body = "\n".join(html.unescape(str(item["ground_truth"])).splitlines()[:-1])
        steps = [to_fraction(x) for x in re.findall(NUMBER, body)]
        steps = [x for x in steps if x is not None and x != q]
        if q is None or q == 0 or learner is None or learner == q or not steps:
            continue
        for transformation, candidate, gold in make_candidates(q, steps[-1], learner):
            reference = fmt(q)
            result = sava_verify(candidate, reference)
            records.append({"reference_id": n_refs, "reference": reference, "transformation": transformation,
                            "candidate": candidate, "gold": gold, "sava": result["verdict"],
                            "sava_stage": result["stage"], "numeric_regex": numeric_regex(reference, candidate),
                            "exact_string": exact_string(reference, candidate), "always_correct": "correct"})
        n_refs += 1
        if n_refs == SAVA_N_REFERENCES:
            break
    sava_benchmark_df = pd.DataFrame(records)
    assert n_refs == SAVA_N_REFERENCES, f"Only {n_refs} eligible references"
    assert (sava_benchmark_df.gold == "correct").sum() == (sava_benchmark_df.gold == "incorrect").sum()
    sava_benchmark_df.to_csv(RUN_ROOT / "sava_constructed_benchmark.csv", index=False)

    methods = ["sava", "numeric_regex", "exact_string", "always_correct"]
    summary = []
    for method in methods:
        right = sava_benchmark_df[method] == sava_benchmark_df.gold
        spoken = sava_benchmark_df[method] != "undetermined"
        tpr = right[sava_benchmark_df.gold == "correct"].mean()
        tnr = right[sava_benchmark_df.gold == "incorrect"].mean()
        summary.append({"method": method, "n": len(right), "coverage": spoken.mean(), "accuracy_all": right.mean(),
                        "balanced_accuracy": (tpr + tnr) / 2,
                        "accuracy_when_spoken": right[spoken].mean() if spoken.any() else float("nan")})
    sava_summary_df = pd.DataFrame(summary)
    sava_ok = (sava_benchmark_df.sava == sava_benchmark_df.gold).to_numpy()
    regex_ok = (sava_benchmark_df.numeric_regex == sava_benchmark_df.gold).to_numpy()
    only_sava, only_regex = int((sava_ok & ~regex_ok).sum()), int((~sava_ok & regex_ok).sum())
    mcnemar_p = binomtest(only_regex, only_sava + only_regex, 0.5).pvalue if only_sava + only_regex else 1.0
    rng = np.random.default_rng(PRIMARY_SEED)
    ids = sava_benchmark_df.reference_id.to_numpy()
    diff_by_ref = pd.Series(sava_ok.astype(float) - regex_ok.astype(float)).groupby(ids).mean().to_numpy()
    boot = [diff_by_ref[rng.integers(0, len(diff_by_ref), len(diff_by_ref))].mean() for _ in range(N_BOOT)]
    sava_summary_df.to_csv(RUN_ROOT / "sava_summary.csv", index=False)
    per_transformation = sava_benchmark_df.assign(**{m: sava_benchmark_df[m] == sava_benchmark_df.gold for m in methods}) \
        .groupby(["gold", "transformation"])[methods].mean()
    per_transformation.to_csv(RUN_ROOT / "sava_per_transformation.csv")
    stages = pd.crosstab(sava_benchmark_df.transformation, sava_benchmark_df.sava_stage)
    stages.to_csv(RUN_ROOT / "sava_benchmark_stages.csv")
    (RUN_ROOT / "sava_tests.json").write_text(json.dumps({
        "references": n_refs, "discordant_sava_only": only_sava, "discordant_regex_only": only_regex,
        "mcnemar_exact_p": mcnemar_p, "accuracy_delta_vs_regex": float(sava_ok.mean() - regex_ok.mean()),
        "delta_ci95": [float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))]}, indent=2))
    display(sava_summary_df)
    display(per_transformation)
    display(stages)

    real_records = []
    frame = load_split_frame("mathdial", "test")
    with in_repo():
        turns = labelled_turns("mathdial", frame)
    for row in turns:
        result = sava_verify(row["student_text"], row["reference"])
        real_records.append({"dialogue": row["dialogue"], "turn_id": row["turn_id"], "label": row["label"],
                             "final_turn": row["final_turn"], "verdict": result["verdict"], "stage": result["stage"]})
    sava_real_df = pd.DataFrame(real_records)
    sava_real_df.to_csv(RUN_ROOT / "sava_real_turns_mathdial.csv", index=False)
    agreement = []
    for subset, part in [("all", sava_real_df), ("final_turn", sava_real_df[sava_real_df.final_turn]),
                         ("non_final_turn", sava_real_df[~sava_real_df.final_turn])]:
        spoken = part[part.verdict != "undetermined"]
        agreement.append({"subset": subset, "n": len(part), "coverage": len(spoken) / max(len(part), 1),
                          "agreement_when_spoken": ((spoken.verdict == "correct").astype(int) == spoken.label).mean()
                          if len(spoken) else float("nan")})
    sava_agreement_df = pd.DataFrame(agreement)
    sava_agreement_df.to_csv(RUN_ROOT / "sava_real_turn_agreement.csv", index=False)
    display(sava_agreement_df)
    display(pd.crosstab(sava_real_df.stage, sava_real_df.label, margins=True))

## 12. REAL-LOGIT ABLATION — CCMF

CCMF is fitted on validation dialogues and evaluated on untouched test dialogues. Parameters are bounded by construction (`a ∈ [e⁻³, e³]`, `b ∈ [−4, 4]`, `s, g < 0.49`), and each label-update mode is **fitted separately**:

| Variant | Label used by Eq. (6) for later turns | Deployable |
|---|---|---|
| `ccmf_no_update` | none | yes |
| `ccmf_model_update` | the model's own prediction | yes |
| `ccmf_sava_update` | SAVA verdict, no update on abstention (the manuscript's Eq. 6) | yes, MathDial only |
| `ccmf_gold_update` | released correctness label | upper bound |

`ccmf_predict_turn` and `ccmf_commit` are the single implementation shared with the integrated online loop in Section 14. Confidence intervals come from a dialogue-level bootstrap in which every variant is scored on the same resamples, giving paired intervals for the difference to `mean_raw`.

In [ ]:
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

CCMF_STARTS = [
    np.array([0.0, 0.0, 0.0, -2.0, -3.0, -0.4, -2.0, -2.0]),
    np.array([0.3, -0.2, -0.5, -1.0, -2.5, 0.0, -1.5, -2.5]),
    np.array([-0.3, 0.2, 0.5, -2.5, -3.5, -1.0, -2.5, -1.5]),
    np.array([0.0, 0.0, -1.0, -0.5, -2.0, -0.8, -1.0, -3.0]),
    np.array([0.6, -0.4, 1.0, -3.0, -4.0, 0.5, -3.0, -1.0]),
]

def unpack_ccmf(raw):
    a = float(np.exp(np.clip(raw[0], -3, 3)))
    b = float(4 * np.tanh(raw[1]))
    lam, mu, p_learn, m0 = (float(x) for x in expit(raw[2:6]))
    s, g = (float(x) for x in 0.49 * expit(raw[6:8]))
    return a, b, lam, mu, p_learn, m0, s, g

def ccmf_predict_turn(state, kcs, gaps, params):
    a, b, lam, mu, p_learn, m0, s, g = params
    observations = expit(a * np.asarray(gaps, dtype=float) + b)
    current, values = {}, []
    for kc, obs in zip(kcs, observations):
        previous = state.get(kc, m0)
        prior = previous + (1 - previous) * p_learn
        value = (1 - lam) * prior + lam * obs
        current[kc] = value
        values.append(value)
    product = float(np.prod(values))
    return (1 - s) * product + g * (1 - product), current

def ccmf_commit(state, current, update_label, mu):
    for kc, value in current.items():
        state[kc] = value if update_label is None else (1 - mu) * value + mu * update_label

def sava_labels(rows):
    return [sava_verify(r["student_text"], r["reference"])["y"] for r in rows]

def predict_ccmf(rows, raw, mode, verdicts=None):
    params = unpack_ccmf(raw)
    state, current_dialogue, predictions = {}, None, []
    for i, row in enumerate(rows):
        if row["dialogue"] != current_dialogue:
            state, current_dialogue = {}, row["dialogue"]
        prediction, current = ccmf_predict_turn(state, row["kcs"], row["gaps"], params)
        predictions.append(prediction)
        label = {"none": None, "model": prediction, "gold": row["label"],
                 "sava": verdicts[i] if verdicts is not None else None}[mode]
        ccmf_commit(state, current, label, params[3])
    return np.clip(np.array(predictions), 1e-6, 1 - 1e-6)

def fit_ccmf(rows, mode, verdicts=None):
    y = np.array([r["label"] for r in rows])
    objective = lambda raw: log_loss(y, predict_ccmf(rows, raw, mode, verdicts), labels=[0, 1])
    fits = [minimize(objective, start, method="Nelder-Mead",
                     options={"maxiter": CCMF_MAXITER, "xatol": 1e-4, "fatol": 1e-6})
            for start in CCMF_STARTS[:CCMF_RESTARTS]]
    best = min(fits, key=lambda fit: fit.fun)
    return best.x, [{"success": bool(f.success), "loss": float(f.fun), "iterations": int(f.nit)} for f in fits]

def fit_platt(rows):
    x = np.concatenate([r["gaps"] for r in rows]).reshape(-1, 1)
    y = np.concatenate([np.repeat(r["label"], len(r["gaps"])) for r in rows])
    if len(np.unique(y)) < 2:
        return 1.0, 0.0
    model = LogisticRegression(C=1e6, max_iter=2000, random_state=PRIMARY_SEED).fit(x, y)
    return float(model.coef_[0, 0]), float(model.intercept_[0])

def fit_noisy_and(rows, a, b):
    y = np.array([r["label"] for r in rows])
    products = np.array([np.prod(expit(a * r["gaps"] + b)) for r in rows])
    def objective(raw):
        s, g = expit(raw) * 0.49
        return log_loss(y, np.clip((1 - s) * products + g * (1 - products), 1e-6, 1 - 1e-6), labels=[0, 1])
    opt = minimize(objective, np.array([-2.0, -2.0]), method="Nelder-Mead", options={"maxiter": 1000})
    return tuple(float(x) for x in expit(opt.x) * 0.49)

def paired_bootstrap(y, predictions, dialogues, reference="mean_raw"):
    rng = np.random.default_rng(PRIMARY_SEED)
    groups = {d: np.flatnonzero(dialogues == d) for d in np.unique(dialogues)}
    ids = list(groups)
    aucs = {k: [] for k in predictions}
    deltas = {k: [] for k in predictions}
    for _ in range(N_BOOT):
        idx = np.concatenate([groups[ids[j]] for j in rng.integers(0, len(ids), len(ids))])
        if len(np.unique(y[idx])) < 2:
            continue
        base = roc_auc_score(y[idx], predictions[reference][idx])
        for key, pred in predictions.items():
            value = roc_auc_score(y[idx], pred[idx])
            aucs[key].append(value)
            deltas[key].append(value - base)
    q = lambda v: (float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975))) if v else (float("nan"),) * 2
    return {k: {"ci": q(aucs[k]), "delta_ci": q(deltas[k])} for k in predictions}

ccmf_rows, CCMF_FITS = [], {}
CCMF_RUNS = [(m, s) for m, s in MODEL_RUNS if m == "qlora" or CCMF_ON_ZERO_SHOT]
if RUN_CCMF:
    for dataset in FIT_ORDER:
        for model_key, seed in CCMF_RUNS:
            tag = f"{dataset}_{model_key}_{seed_tag(model_key, seed)}"
            fit_on = FIT_ON[dataset]
            reuse = fit_on != dataset          # CoMTA evaluation only: parameters fitted on MathDial validation
            val_rows = load_rows(fit_on, model_key, seed, "val")
            test_rows = load_rows(dataset, model_key, seed, "test")
            y_test = np.array([r["label"] for r in test_rows])
            dialogues = np.array([r["dialogue"] for r in test_rows])
            if reuse:
                fitted = CCMF_FITS[(fit_on, model_key, seed)]
                a, b, s, g = fitted["a"], fitted["b"], fitted["s"], fitted["g"]
            else:
                a, b = fit_platt(val_rows)
                s, g = fit_noisy_and(val_rows, a, b)
                fitted = CCMF_FITS[(dataset, model_key, seed)] = {"a": a, "b": b, "s": s, "g": g, "raw": {}}
            products = np.array([np.prod(expit(a * r["gaps"] + b)) for r in test_rows])
            predictions = {
                "mean_raw": mean_prediction(test_rows),
                "mean_platt": np.array([expit(a * r["gaps"] + b).mean() for r in test_rows]),
                "noisy_and_platt": (1 - s) * products + g * (1 - products),
            }
            modes = ["none", "model", "gold"] + (["sava"] if dataset == "mathdial" else [])
            parameters = {"fit_on": fit_on, "platt": {"a": a, "b": b}, "noisy_and": {"s": s, "g": g}}
            fit_logs = {}
            for mode in modes:
                test_verdicts = sava_labels(test_rows) if mode == "sava" else None
                if reuse:
                    raw, fits = fitted["raw"][mode], []
                else:
                    val_verdicts = sava_labels(val_rows) if mode == "sava" else None
                    raw, fits = fit_ccmf(val_rows, mode, val_verdicts)
                    fitted["raw"][mode] = raw
                name = {"none": "ccmf_no_update", "model": "ccmf_model_update",
                        "gold": "ccmf_gold_update", "sava": "ccmf_sava_update"}[mode]
                predictions[name] = predict_ccmf(test_rows, raw, mode, test_verdicts)
                keys = ["a", "b", "lambda", "mu", "p_learn", "m0", "slip", "guess"]
                parameters[name] = {"raw": [float(x) for x in raw], **dict(zip(keys, unpack_ccmf(raw)))}
                fit_logs[name] = fits
                print(tag, name, "restarts:", [(f["success"], round(f["loss"], 5)) for f in fits])
            boot = paired_bootstrap(y_test, predictions, dialogues)
            for name, pred in predictions.items():
                ccmf_rows.append({"dataset": dataset, "model": model_key, "seed": seed_tag(model_key, seed),
                                  "variant": name, "n": len(y_test), "auc": safe_auc(y_test, pred),
                                  "ci_low": boot[name]["ci"][0], "ci_high": boot[name]["ci"][1],
                                  "delta_vs_mean_raw": safe_auc(y_test, pred) - safe_auc(y_test, predictions["mean_raw"]),
                                  "delta_ci_low": boot[name]["delta_ci"][0], "delta_ci_high": boot[name]["delta_ci"][1]})
            (RUN_ROOT / f"ccmf_parameters_{tag}.json").write_text(json.dumps({"parameters": parameters, "fits": fit_logs}, indent=2))
            pd.DataFrame({"dialogue": dialogues, "turn_id": [r["turn_id"] for r in test_rows], "label": y_test,
                          **predictions}).to_csv(RUN_ROOT / f"ccmf_test_predictions_{tag}.csv", index=False)

ccmf_df = pd.DataFrame(ccmf_rows)
ccmf_df.to_csv(RUN_ROOT / "ccmf_real_logits_ablation.csv", index=False)
display(ccmf_df)

## 13. SIMULATED DATA — Maintain-Then-Advance (MTA) Planner

**Why the previous planner failed.** Lowest-mastery-first selection spreads practice across every weak KC; with forgetting, nothing crosses τ. The previous prerequisite-aware variant kept that rule inside the ready set and therefore inherited the failure.

**MTA** turns the mastery state into three ordered rules:

1. **Maintain** — a KC that was mastered and has slipped below τ is restored first, nearest to τ first (cheapest to recover).
2. **Commit** — otherwise keep practising the current frontier KC until it reaches τ.
3. **Advance** — a new frontier KC must have all prerequisites mastered; among them pick the one nearest to mastery (shortest remaining work).

MTA needs only the prerequisite graph and the mastery estimate, not a hand-authored curriculum order. Design choices were fixed on development learner seeds 10000–10059; every number below uses disjoint evaluation seeds starting at 221.

**Baselines.** greedy lowest-first, random, the previous prerequisite-aware lowest-first planner, a fixed curriculum without skipping (previous paper), and two **filtered curricula** that skip mastered KCs — one in index order (a perfect topological order in this simulator) and one in a random valid topological order. **Ablations** remove maintenance, replace nearest-first by lowest-first, or ignore prerequisites.

**Sensitivity grid.** graph ∈ {three chains, random DAG} × prior knowledge ∈ {uniform, 30 % partially known} × forgetting ∈ {0, 0.002, 0.004, 0.008} × readiness ∈ {1.0, 0.8, 0.6} × estimator noise σ ∈ {0, 0.05, 0.10}.

In [ ]:
from dataclasses import dataclass, asdict, replace
from scipy.stats import wilcoxon

@dataclass(frozen=True)
class SimConfig:
    n_kcs: int = 12
    turns: int = 120
    tau: float = 0.8
    gain: float = 0.185
    decay: float = 0.004
    readiness: float = 0.8
    noise: float = 0.0
    graph: str = "chains"
    prior: str = "uniform"

PAPER_CONFIG = SimConfig()
DEV_SEEDS = range(10_000, 10_060)

def make_prerequisites(cfg, rng):
    if cfg.graph == "chains":
        return {k: ([] if k % 4 == 0 else [k - 1]) for k in range(cfg.n_kcs)}
    prerequisites = {0: []}
    for j in range(1, cfg.n_kcs):
        size = min(int(rng.integers(0, 3)), j)
        prerequisites[j] = sorted(int(x) for x in rng.choice(j, size=size, replace=False))
    return prerequisites

def random_topological_order(prerequisites, rng):
    remaining, order = set(prerequisites), []
    while remaining:
        ready = sorted(k for k in remaining if all(p not in remaining for p in prerequisites[k]))
        k = int(rng.choice(ready))
        order.append(k)
        remaining.remove(k)
    return order

def mta_choose(obs, prerequisites, memory, tau, maintain=True, nearest=True, use_prerequisites=True):
    ever = memory.setdefault("ever", set())
    ever.update(int(k) for k in np.flatnonzero(obs >= tau))
    if maintain:
        slipped = [k for k in sorted(ever) if obs[k] < tau]
        if slipped:
            memory["current"] = max(slipped, key=lambda k: obs[k])
            return memory["current"]
    current = memory.get("current")
    if current is not None and current not in ever:
        return current
    todo = [k for k in range(len(obs)) if k not in ever]
    if not todo:
        below = [k for k in range(len(obs)) if obs[k] < tau]
        return max(below, key=lambda k: obs[k]) if below else None
    ready = [k for k in todo if not use_prerequisites or all(p in ever for p in prerequisites.get(k, []))]
    pool = ready or todo
    memory["current"] = max(pool, key=lambda k: obs[k]) if nearest else min(pool, key=lambda k: obs[k])
    return memory["current"]

def unmastered(obs, tau):
    return np.flatnonzero(obs < tau)

POLICIES = {
    "greedy_lowest_first": lambda o, pre, mem, rng, t, cfg: (lambda u: None if len(u) == 0 else int(u[np.argmin(o[u])]))(unmastered(o, cfg.tau)),
    "random_unmastered": lambda o, pre, mem, rng, t, cfg: (lambda u: None if len(u) == 0 else int(rng.choice(u)))(unmastered(o, cfg.tau)),
    "fixed_curriculum_no_skip": lambda o, pre, mem, rng, t, cfg: None if len(unmastered(o, cfg.tau)) == 0 else int(t % cfg.n_kcs),
    "prereq_lowest_first_previous": lambda o, pre, mem, rng, t, cfg: (lambda u: None if len(u) == 0 else (lambda pool: int(pool[np.argmin(o[pool])]))(
        [k for k in u if all(o[p] >= cfg.tau for p in pre[k])] or list(u)))(unmastered(o, cfg.tau)),
    "filtered_curriculum_index_order": lambda o, pre, mem, rng, t, cfg: (lambda u: None if len(u) == 0 else int(u[0]))(unmastered(o, cfg.tau)),
    "filtered_curriculum_random_topo": lambda o, pre, mem, rng, t, cfg: next((k for k in mem["order"] if o[k] < cfg.tau), None),
    "mta": lambda o, pre, mem, rng, t, cfg: mta_choose(o, pre, mem, cfg.tau),
    "ablation_no_maintain": lambda o, pre, mem, rng, t, cfg: mta_choose(o, pre, mem, cfg.tau, maintain=False),
    "ablation_lowest_first": lambda o, pre, mem, rng, t, cfg: mta_choose(o, pre, mem, cfg.tau, nearest=False),
    "ablation_no_prerequisites": lambda o, pre, mem, rng, t, cfg: mta_choose(o, pre, mem, cfg.tau, use_prerequisites=False),
}
MAIN_POLICIES = ["greedy_lowest_first", "random_unmastered", "fixed_curriculum_no_skip", "prereq_lowest_first_previous",
                 "filtered_curriculum_index_order", "filtered_curriculum_random_topo", "mta"]
ABLATIONS = ["mta", "ablation_no_maintain", "ablation_lowest_first", "ablation_no_prerequisites"]

def simulate_planner(policy, learner_seed, cfg=PAPER_CONFIG):
    rng = np.random.default_rng(learner_seed)
    observation_rng = np.random.default_rng(learner_seed + 1_000_000)
    graph_rng = np.random.default_rng(learner_seed + 2_000_000)
    if cfg.prior == "uniform":
        mastery = rng.uniform(0.05, 0.40, cfg.n_kcs)
    else:
        mastery = np.where(rng.random(cfg.n_kcs) < 0.3, rng.uniform(0.60, 0.95, cfg.n_kcs), rng.uniform(0.05, 0.40, cfg.n_kcs))
    difficulty = rng.uniform(0.7, 1.3, cfg.n_kcs)
    prerequisites = make_prerequisites(cfg, graph_rng)
    memory = {"order": random_topological_order(prerequisites, np.random.default_rng(learner_seed + 3_000_000))}
    for turn in range(cfg.turns):
        obs = np.clip(mastery + observation_rng.normal(0, cfg.noise, cfg.n_kcs), 0, 1) if cfg.noise else mastery.copy()
        kc = POLICIES[policy](obs, prerequisites, memory, rng, turn, cfg)
        if kc is None:
            break
        readiness = 1.0 if all(mastery[p] >= cfg.tau for p in prerequisites[kc]) else cfg.readiness
        gain = (cfg.gain / difficulty[kc]) * readiness * (1 - mastery[kc])
        mastery *= (1 - cfg.decay)
        mastery[kc] = min(1.0, mastery[kc] + gain)
    return float(np.mean(mastery >= cfg.tau))

EVAL_SEEDS = [PRIMARY_SEED + i for i in range(PLANNER_LEARNERS)]
GRID_SEEDS = [PRIMARY_SEED + i for i in range(PLANNER_GRID_LEARNERS)]
assert not set(EVAL_SEEDS) & set(DEV_SEEDS)

def evaluate_policies(policies, cfg, seeds):
    return {p: np.array([simulate_planner(p, s, cfg) for s in seeds]) for p in policies}

def summarize(values, reference="mta"):
    rng = np.random.default_rng(PRIMARY_SEED)
    rows = []
    for policy, v in values.items():
        boot = [rng.choice(v, len(v), replace=True).mean() for _ in range(2000)]
        diff = values[reference] - v
        p_value = wilcoxon(values[reference], v).pvalue if policy != reference and np.any(diff) else float("nan")
        rows.append({"policy": policy, "n_learners": len(v), "fraction_mastered": v.mean(),
                     "ci_low": np.quantile(boot, 0.025), "ci_high": np.quantile(boot, 0.975),
                     "mta_minus_policy": diff.mean(), "wilcoxon_p_vs_mta": p_value})
    return pd.DataFrame(rows)

planner_df = pd.DataFrame()
planner_ablation_df = pd.DataFrame()
planner_grid_df = pd.DataFrame()
if RUN_PLANNER:
    planner_df = summarize(evaluate_policies(MAIN_POLICIES, PAPER_CONFIG, EVAL_SEEDS))
    planner_df.to_csv(RUN_ROOT / "planner_simulation.csv", index=False)
    display(planner_df)
    planner_ablation_df = summarize(evaluate_policies(ABLATIONS, PAPER_CONFIG, EVAL_SEEDS))
    planner_ablation_df.to_csv(RUN_ROOT / "planner_ablation.csv", index=False)
    display(planner_ablation_df)

    grid_rows = []
    grid = [replace(PAPER_CONFIG, graph=g, prior=pr, decay=dc, readiness=rd, noise=nz)
            for g in ["chains", "dag"] for pr in ["uniform", "partial"] for dc in [0.0, 0.002, 0.004, 0.008]
            for rd in [1.0, 0.8, 0.6] for nz in [0.0, 0.05, 0.10]]
    for cfg in grid:
        values = evaluate_policies(MAIN_POLICIES, cfg, GRID_SEEDS)
        for policy, v in values.items():
            grid_rows.append({**asdict(cfg), "policy": policy, "fraction_mastered": v.mean()})
    planner_grid_df = pd.DataFrame(grid_rows)
    planner_grid_df.to_csv(RUN_ROOT / "planner_sensitivity_grid.csv", index=False)
    keys = ["graph", "prior", "decay", "readiness", "noise"]
    wide = planner_grid_df.pivot_table(index=keys, columns="policy", values="fraction_mastered")
    wins = pd.DataFrame({policy: {"mta_at_least_as_good": int((wide["mta"] >= wide[policy] - 1e-9).sum()),
                                  "cells": len(wide), "mean_mta_minus_policy": float((wide["mta"] - wide[policy]).mean())}
                         for policy in MAIN_POLICIES if policy != "mta"}).T
    wins.to_csv(RUN_ROOT / "planner_grid_wins.csv")
    display(wins)
    display(wide.groupby(level=["graph", "noise"]).mean().round(3))

## 14. REAL DATA — integrated online loop

This is the runtime order of the architecture, executed on every MathDial test turn with the fitted SAVA-mode CCMF parameters:

1. QLoRA Mastery Estimator logit gaps → `ccmf_predict_turn` predicts correctness **before** the answer is judged;
2. `sava_verify` judges the learner's answer; an incorrect verdict opens the Error Analyzer gate;
3. `ccmf_commit` updates mastery with the verdict (no update on abstention), for later turns only;
4. MTA selects the next KC from the updated state. The Common Core KC tags carry no released prerequisite graph, so the loop uses an empty graph and rules 1–3 of MTA reduce to maintain/commit/nearest-first.

The loop must reproduce `ccmf_sava_update` from Section 12 exactly; that assertion ties the ablation numbers to the deployed control flow.

In [ ]:
integrated_df = pd.DataFrame()
INTEGRATED_TARGET = ("qlora", PRIMARY_SEED) if RUN_QLORA else ("zero_shot", None)
if RUN_INTEGRATED and RUN_CCMF and RUN_SAVA and "mathdial" in DATASETS:
    model_key, seed = INTEGRATED_TARGET
    tag = f"mathdial_{model_key}_{seed_tag(model_key, seed)}"
    stored = json.loads((RUN_ROOT / f"ccmf_parameters_{tag}.json").read_text())["parameters"]["ccmf_sava_update"]
    params = unpack_ccmf(np.array(stored["raw"]))
    rows = load_rows("mathdial", model_key, seed, "test")
    records, state, planner_memory, current_dialogue = [], {}, {}, None
    for row in rows:
        if row["dialogue"] != current_dialogue:
            state, planner_memory, current_dialogue = {}, {}, row["dialogue"]
        predicted, current = ccmf_predict_turn(state, row["kcs"], row["gaps"], params)
        verdict = sava_verify(row["student_text"], row["reference"])
        ccmf_commit(state, current, verdict["y"], params[3])
        # MTA works on indices; planner memory is kept by KC name because the KC set grows over a dialogue.
        names = sorted(state)
        position = {kc: i for i, kc in enumerate(names)}
        memory = {"ever": {position[kc] for kc in planner_memory.get("ever", set())},
                  "current": position.get(planner_memory.get("current"))}
        next_index = mta_choose(np.array([state[k] for k in names]), {}, memory, PAPER_CONFIG.tau)
        planner_memory = {"ever": {names[i] for i in memory["ever"]},
                          "current": names[memory["current"]] if memory.get("current") is not None else None}
        records.append({"dialogue": row["dialogue"], "turn_id": row["turn_id"], "label": row["label"],
                        "final_turn": row["final_turn"], "predicted_correctness": float(np.clip(predicted, 1e-6, 1 - 1e-6)),
                        "sava_verdict": verdict["verdict"], "sava_stage": verdict["stage"],
                        "diagnosis_gate": "run_error_analyzer" if verdict["verdict"] == "incorrect" else "skip_error_analyzer",
                        "n_kcs": len(row["kcs"]), "kcs": json.dumps(row["kcs"]),
                        "kc_mastery_before": json.dumps(current), "sava_extracted": verdict["extracted"],
                        "mean_mastery_after": float(np.mean([state[k] for k in current])),
                        "planner_next_kc": names[next_index] if next_index is not None else None})
    integrated_df = pd.DataFrame(records)
    integrated_df.to_csv(RUN_ROOT / f"integrated_run_{tag}.csv", index=False)
    reference = pd.read_csv(RUN_ROOT / f"ccmf_test_predictions_{tag}.csv")["ccmf_sava_update"].to_numpy()
    assert len(integrated_df) == len(reference)
    assert np.allclose(integrated_df.predicted_correctness.to_numpy(), reference, atol=1e-9)
    print("Integrated loop reproduces ccmf_sava_update on", len(integrated_df), "turns.")
    display(integrated_df.head(10))
    display(integrated_df.groupby(["sava_verdict", "diagnosis_gate"]).size().rename("turns"))

## 15. REAL DATA — Error Analyzer (LLM agent)

With `PEDAGOGY_BACKEND = "local"` (default) the Error Analyzer and Feedback Generator run on the same Llama-3.1-8B-Instruct backbone, loaded in 4-bit NF4 with greedy decoding; with `"openrouter"` they call `PEDAGOGY_OPENROUTER_MODEL` through OpenRouter at temperature 0 (same prompts, validator and fallback). By default the base instruct weights are used, because the QLoRA adapter was trained only to emit `True`/`False` after KC prompts; set `PEDAGOGY_USE_KT_ADAPTER = True` to generate with the KT adapter attached as an ablation. Every output must pass a JSON schema validator; an invalid output falls back to the rule-based analyzer and is counted.

Schema: `error_category` ∈ {conceptual, procedural, calculation, careless, other}, `explanation`, `affected_concepts`, `severity` ∈ {low, medium, high}, `instructional_suggestions`.

Two item sets:

* `comta_incorrect_label` — every CoMTA test turn whose released correctness label is incorrect (first labelled turns included). This set measures **categorisation, not detection**, and is the one scored against human gold labels.
* `mastery-aware, SAVA-gated` MathDial turns — the Error Analyzer as deployed in Section 14: triggered by an incorrect SAVA verdict, with per-KC mastery from CCMF.

**Gold labels.** Upload `/content/error_analyzer_gold.csv` with columns `dialogue, turn_id, gold` and optionally `gold_secondary` (a second acceptable category when conceptual and procedural genuinely overlap). Without it, the cell writes a **blind** annotation template that contains the dialogue context but no model predictions. Metrics: exact match, macro-F1 over the five classes, and relaxed agreement (prediction equals `gold` or `gold_secondary`), compared with the rule-based analyzer and a majority-class baseline.

In [ ]:
import gc
from sklearn.metrics import f1_score as sk_f1_score

ERROR_CLASSES = ["conceptual", "procedural", "calculation", "careless", "other"]
SEVERITIES = ["low", "medium", "high"]
PEDAGOGY_SYSTEM = "You are a mathematics tutor. Reply with a single JSON object only."
GEN_MODEL = globals().get("GEN_MODEL")
GEN_TOKENIZER = globals().get("GEN_TOKENIZER")
GEN_ADAPTER_NAME = globals().get("GEN_ADAPTER_NAME")
ea_df = pd.DataFrame()
ea_metrics_df = pd.DataFrame()

def load_generator():
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")
    tok.pad_token = tok.eos_token
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                      bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quantization,
                                                 dtype=torch.float16, device_map={"": 0})
    adapter = None
    if PEDAGOGY_USE_KT_ADAPTER and ADAPTERS:
        from peft import PeftModel
        key = ("mathdial", PRIMARY_SEED) if ("mathdial", PRIMARY_SEED) in ADAPTERS else next(iter(ADAPTERS))
        adapter = ADAPTERS[key]
        model = PeftModel.from_pretrained(model, str(REPO / "saved_models" / adapter))
    model.eval()
    return model, tok, adapter

import urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor

def openrouter_key():
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENROUTER_API_KEY")

def openrouter_post(payload, retries=4):
    key = openrouter_key()
    assert key, "Add OPENROUTER_API_KEY to Colab Secrets (key icon in the left sidebar) and enable notebook access."
    request = urllib.request.Request(OPENROUTER_URL, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}",
                                              "X-Title": "SoICT2026 replication"})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(request, timeout=180) as response:
                return json.loads(response.read())["choices"][0]["message"]["content"] or ""
        except urllib.error.HTTPError as error:
            if error.code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                time.sleep(5 * 2 ** attempt)
                continue
            raise RuntimeError(f"OpenRouter HTTP {error.code}: {error.read()[:300]!r}") from None
        except urllib.error.URLError:
            if attempt < retries - 1:
                time.sleep(5 * 2 ** attempt)
                continue
            raise

def generate_texts(prompts):
    if PEDAGOGY_BACKEND == "openrouter":
        def one(prompt):
            return openrouter_post({"model": PEDAGOGY_OPENROUTER_MODEL, "temperature": 0, "max_tokens": GEN_MAX_NEW_TOKENS,
                                    "messages": [{"role": "system", "content": PEDAGOGY_SYSTEM},
                                                 {"role": "user", "content": prompt}]})
        with ThreadPoolExecutor(max_workers=4) as pool:
            outputs = list(pool.map(one, prompts))
        print(f"generated {len(outputs)}/{len(prompts)} via OpenRouter ({PEDAGOGY_OPENROUTER_MODEL})")
        return outputs
    assert PEDAGOGY_BACKEND == "local", PEDAGOGY_BACKEND
    return generate_texts_local(prompts)

def generate_texts_local(prompts):
    outputs = []
    for start in range(0, len(prompts), GEN_BATCH_SIZE):
        batch = prompts[start:start + GEN_BATCH_SIZE]
        chats = [GEN_TOKENIZER.apply_chat_template([{"role": "system", "content": PEDAGOGY_SYSTEM},
                                                    {"role": "user", "content": p}],
                                                   tokenize=False, add_generation_prompt=True) for p in batch]
        encoded = GEN_TOKENIZER(chats, return_tensors="pt", padding=True, add_special_tokens=False).to(GEN_MODEL.device)
        with torch.inference_mode():
            generated = GEN_MODEL.generate(**encoded, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False,
                                           temperature=None, top_p=None, pad_token_id=GEN_TOKENIZER.pad_token_id)
        for row in generated[:, encoded["input_ids"].shape[1]:]:
            outputs.append(GEN_TOKENIZER.decode(row, skip_special_tokens=True))
        print(f"generated {len(outputs)}/{len(prompts)}")
    return outputs

def parse_json_object(text):
    match = re.search(r"\{.*\}", str(text), flags=re.S)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def short_kc(kc, limit=160):
    kc = str(kc)
    return kc if len(kc) <= limit else kc[:limit - 3] + "..."

def validate_error_analysis(obj):
    if not obj:
        return None
    category = str(obj.get("error_category", "")).strip().lower()
    severity = str(obj.get("severity", "")).strip().lower()
    explanation, concepts, suggestions = obj.get("explanation"), obj.get("affected_concepts"), obj.get("instructional_suggestions")
    if category not in ERROR_CLASSES or severity not in SEVERITIES:
        return None
    if not isinstance(explanation, str) or not explanation.strip():
        return None
    if not isinstance(concepts, list) or not isinstance(suggestions, list) or not suggestions:
        return None
    return {"error_category": category, "explanation": explanation.strip(), "severity": severity,
            "affected_concepts": [str(c) for c in concepts], "instructional_suggestions": [str(s) for s in suggestions]}

def error_analyzer_prompt(item):
    mastery = ", ".join(f"{short_kc(k, 60)}: {v:.2f}" for k, v in item["mastery"].items()) or "not available"
    return (
        "Analyse the student's error in a mathematics tutoring dialogue.\n"
        f"Problem: {item.get('problem') or 'not provided'}\n"
        f"Reference answer: {item.get('reference') or 'not provided'}\n"
        f"Dialogue so far:\n{item['history']}\n\n"
        f"Student turn to analyse: {item['student_text']}\n"
        f"Extracted student expression (symbolic verifier): {item.get('extracted')}\n"
        f"Knowledge components involved: {'; '.join(short_kc(k) for k in item['kcs'])}\n"
        f"Current mastery estimates: {mastery}\n\n"
        "Category definitions: conceptual = misunderstanding of a concept or of the relation between quantities; "
        "procedural = a wrong, missing or misordered step of a method; calculation = an arithmetic slip inside an "
        "otherwise correct method; careless = a copying, sign or reading slip; other = none of these.\n"
        "Return JSON with exactly these keys: error_category (one of conceptual, procedural, calculation, careless, other), "
        "explanation (string), affected_concepts (list of KC names from the list above), severity (low|medium|high), "
        "instructional_suggestions (list of 2-3 strings)."
    )

def rule_based_error_analysis(item):
    # Port of kse2026_code/pedagogy.py ErrorAnalyzer.rule_based
    text = str(item["student_text"]).strip().lower()
    extracted, reference = item.get("extracted"), item.get("reference_parsed")
    mastery, kcs = item["mastery"], item["kcs"]
    category, explanation = "other", "The error could not be attributed to a specific mechanism from the text alone."
    try:
        a, b = float(extracted), float(reference)
        if abs(a + b) < 1e-9 and b != 0:
            category, explanation = "careless", f"The answer {a:g} has the right magnitude but the wrong sign."
        elif abs(a - b) <= 1:
            category, explanation = "careless", f"The answer {a:g} is off by {abs(a - b):g} from {b:g}."
        elif b != 0 and (abs(a / b - 2) < 1e-9 or abs(a / b - 0.5) < 1e-9):
            category, explanation = "procedural", f"The answer {a:g} is {b:g} scaled by two: a step was applied or omitted."
        elif b != 0 and abs(a * b - 1) < 1e-9:
            category, explanation = "conceptual", f"The answer {a:g} is the reciprocal of {b:g}: the relation was inverted."
        elif b != 0 and abs(a - b) / max(1.0, abs(b)) < 0.15:
            category, explanation = "calculation", f"The answer {a:g} is close to {b:g}: an arithmetic slip."
        else:
            category, explanation = "procedural", f"The answer {a:g} differs substantially from {b:g}."
    except (TypeError, ValueError):
        symbolic_ext = extracted is not None and re.search(r"[a-z]", str(extracted)) is not None
        symbolic_ref = reference is not None and re.search(r"[a-z]", str(reference)) is not None
        if symbolic_ext and symbolic_ref:
            category, explanation = "procedural", "A rule was applied to only part of the expression."
        elif symbolic_ext:
            category, explanation = "procedural", "The student stopped at an intermediate algebraic form."
        elif any(w in text for w in ("because", "since", "means", "should", "rule", "always")):
            category, explanation = "conceptual", "The explanation reveals a misconception about the underlying rule."
    weakest = min((mastery.get(k, 0.5) for k in kcs), default=0.5)
    if category in ("careless", "calculation"):
        severity = "low" if weakest >= 0.5 else "medium"
    else:
        severity = "high" if weakest < 0.4 else "medium"
    affected = [k for k in kcs if mastery.get(k, 0.5) < 0.7] or list(kcs)
    return {"error_category": category, "explanation": explanation, "severity": severity,
            "affected_concepts": affected, "instructional_suggestions": ["Ask the student to explain the step that produced this answer."]}

def run_error_analyzer(items):
    outputs = generate_texts([error_analyzer_prompt(item) for item in items]) if items else []
    records = []
    for item, text in zip(items, outputs):
        parsed = validate_error_analysis(parse_json_object(text))
        rule = rule_based_error_analysis(item)
        final = parsed or rule
        records.append({"set": item["set"], "dataset": item["dataset"], "dialogue": item["dialogue"],
                        "turn_id": item["turn_id"], "label": item["label"], "sava_verdict": item["sava_verdict"],
                        "kcs": json.dumps(item["kcs"]), "valid_json": parsed is not None,
                        "source": "llm" if parsed else "rule_fallback", "category": final["error_category"],
                        "severity": final["severity"], "explanation": final["explanation"],
                        "suggestions": json.dumps(final["instructional_suggestions"]),
                        "category_rule_based": rule["error_category"], "raw_output": str(text)[:4000]})
    return pd.DataFrame(records)

def export_mastery_lookup(dataset):
    model_key, seed = ("qlora", PRIMARY_SEED) if RUN_QLORA else ("zero_shot", None)
    path = rows_path(dataset, model_key, seed, "test")
    if not path.exists():
        return {}
    return {(r["dialogue"], r["turn_id"]): {kc: float(1 / (1 + np.exp(-g))) for kc, g in zip(r["kcs"], r["gaps"])}
            for r in load_rows(dataset, model_key, seed, "test")}

print("Error Analyzer definitions ready.")

In [ ]:
EA_ITEMS = []
if RUN_ERROR_ANALYZER and "comta" in DATASETS:
    frame = load_split_frame("comta", "test")
    with in_repo():
        turns = labelled_turns("comta", frame, skip_first_turn=False, with_context=True)
    lookup = export_mastery_lookup("comta")
    incorrect = [t for t in turns if t["label"] == 0]
    for t in incorrect[:EA_MAX_COMTA_TURNS] if EA_MAX_COMTA_TURNS else incorrect:
        verdict = sava_verify(t["student_text"], t["reference"])
        EA_ITEMS.append({**t, "set": "comta_incorrect_label", "dataset": "comta", "sava_verdict": verdict["verdict"],
                         "extracted": verdict["extracted"], "reference_parsed": verdict["reference"],
                         "mastery": lookup.get((t["dialogue"], t["turn_id"]), {})})
if RUN_ERROR_ANALYZER and not integrated_df.empty:
    frame = load_split_frame("mathdial", "test")
    with in_repo():
        context = {(t["dialogue"], t["turn_id"]): t for t in labelled_turns("mathdial", frame, with_context=True)}
    gated = integrated_df[integrated_df.diagnosis_gate == "run_error_analyzer"]
    if len(gated) > EA_MAX_MATHDIAL_TURNS:
        gated = gated.sample(n=EA_MAX_MATHDIAL_TURNS, random_state=PRIMARY_SEED).sort_index()
    for _, g in gated.iterrows():
        t = context[(str(g.dialogue), int(g.turn_id))]
        verdict = sava_verify(t["student_text"], t["reference"])
        EA_ITEMS.append({**t, "set": "mathdial_sava_gated", "dataset": "mathdial", "sava_verdict": verdict["verdict"],
                         "extracted": verdict["extracted"], "reference_parsed": verdict["reference"],
                         "mastery": json.loads(g.kc_mastery_before)})

if PEDAGOGY_BACKEND == "local" and (EA_ITEMS or (RUN_FEEDBACK and not integrated_df.empty)):
    if GEN_MODEL is None:
        GEN_MODEL, GEN_TOKENIZER, GEN_ADAPTER_NAME = load_generator()
if EA_ITEMS:
    ea_df = run_error_analyzer(EA_ITEMS)
    ea_df.to_csv(RUN_ROOT / "error_analyzer_predictions.csv", index=False)
    display(ea_df.groupby("set").agg(n=("valid_json", "size"), valid_json_rate=("valid_json", "mean")))
    display(pd.crosstab(ea_df.set, ea_df.category))

    comta_pred = ea_df[ea_df.set == "comta_incorrect_label"]
    gold_path = WORK / "error_analyzer_gold.csv"
    if gold_path.exists() and not comta_pred.empty:
        gold = pd.read_csv(gold_path, dtype={"dialogue": str})
        assert {"dialogue", "turn_id", "gold"}.issubset(gold.columns)
        gold["gold"] = gold["gold"].astype(str).str.strip().str.lower()
        gold["gold_secondary"] = gold["gold_secondary"].fillna("").astype(str).str.strip().str.lower() \
            if "gold_secondary" in gold.columns else ""
        assert set(gold.gold).issubset(ERROR_CLASSES), set(gold.gold) - set(ERROR_CLASSES)
        merged = comta_pred.merge(gold[["dialogue", "turn_id", "gold", "gold_secondary"]], on=["dialogue", "turn_id"])
        assert len(merged) == len(gold), "Every gold row must match an analysed CoMTA turn."
        majority = merged.gold.value_counts().idxmax()
        rows = []
        for method, prediction in [("llm_error_analyzer", merged.category),
                                   ("llm_valid_json_only", merged.category.where(merged.valid_json)),
                                   ("rule_based", merged.category_rule_based),
                                   ("majority_class", pd.Series(majority, index=merged.index))]:
            keep = prediction.notna()
            p, g, s = prediction[keep], merged.gold[keep], merged.gold_secondary[keep]
            rows.append({"method": method, "n": int(keep.sum()), "exact_match": float((p == g).mean()),
                         "macro_f1": float(sk_f1_score(g, p, labels=ERROR_CLASSES, average="macro", zero_division=0)),
                         "relaxed_agreement": float(((p == g) | (p == s)).mean())})
        ea_metrics_df = pd.DataFrame(rows)
        ea_metrics_df.to_csv(RUN_ROOT / "error_analyzer_metrics.csv", index=False)
        display(ea_metrics_df)
        merged.to_csv(RUN_ROOT / "error_analyzer_gold_merged.csv", index=False)
        cm = confusion_matrix(merged.gold, merged.category, labels=ERROR_CLASSES)
        fig, ax = plt.subplots(figsize=(7, 6))
        ConfusionMatrixDisplay(cm, display_labels=ERROR_CLASSES).plot(ax=ax, cmap="Oranges", colorbar=False, xticks_rotation=35)
        plt.title("Error Analyzer vs human gold (CoMTA incorrect turns)")
        plt.tight_layout()
        plt.savefig(RUN_ROOT / "error_analyzer_confusion_matrix.png", dpi=200)
        plt.show()
    elif not comta_pred.empty:
        items = [i for i in EA_ITEMS if i["set"] == "comta_incorrect_label"]
        pd.DataFrame({"dialogue": [i["dialogue"] for i in items], "turn_id": [i["turn_id"] for i in items],
                      "history_tail": [i["history"][-800:] for i in items], "student_text": [i["student_text"] for i in items],
                      "kcs": [json.dumps(i["kcs"]) for i in items], "gold": "", "gold_secondary": ""}).to_csv(
            RUN_ROOT / "error_analyzer_gold_TEMPLATE_blind.csv", index=False)
        print("No gold file: wrote a blind annotation template (no model predictions) for",
              len(items), "CoMTA turns. Annotate it, upload as /content/error_analyzer_gold.csv and rerun this cell.")

## 16. REAL DATA — Feedback Generator (LLM agent)

Thirty MathDial test turns from the integrated loop are sampled deterministically, ten per SAVA verdict (correct, incorrect, undetermined). Each case gives the generator the dialogue context, the verified correctness, per-KC mastery from CCMF, the Error Analyzer diagnosis for incorrect turns, and the KC selected by MTA. The weakest KC's mastery band sets the requested strategy: low (< 0.4) worked example / re-teaching, medium (0.4–0.7) scaffolded guiding question, high (≥ 0.7) minimal hint / self-explanation.

Schema: `feedback_text, scaffolding_question, mastery_adaptation, pedagogical_strategy, next_step_hint`.

**Automatic checks** (no judge needed): schema validity; **answer leakage** — the reference answer appears in the feedback of a turn that is not verified correct; **strategy–band consistency** — the stated strategy names the approach required by the mastery band; response length.

**LLM-as-judge via OpenRouter.** With `RUN_FEEDBACK_JUDGE = True` and an `OPENROUTER_API_KEY` stored in Colab Secrets, `JUDGE_MODEL` (default `openai/gpt-4o`) scores Correctness, Relevance and Clarity on a 1–5 scale at temperature 0. This sends dialogue excerpts to OpenRouter and the model provider. It is an automated proxy with judge bias and no inter-rater agreement, not human validation. The judge requests are always written to `feedback_judge_requests.jsonl` for audit when the judge is disabled.

In [ ]:
FEEDBACK_KEYS = ["feedback_text", "scaffolding_question", "mastery_adaptation", "pedagogical_strategy", "next_step_hint"]
BAND_KEYWORDS = {"low": ("worked", "re-teach", "reteach", "example", "demonstrat"),
                 "medium": ("scaffold", "guiding", "guided"),
                 "high": ("hint", "self-explanation", "self explanation")}
feedback_df = pd.DataFrame()
feedback_summary_df = pd.DataFrame()

def mastery_band(value):
    return "low" if value < 0.4 else ("medium" if value < 0.7 else "high")

def feedback_prompt(case):
    mastery = ", ".join(f"{short_kc(k, 60)}: {v:.2f}" for k, v in case["mastery"].items()) or "not available"
    diagnosis = json.dumps(case["error_analysis"]) if case["error_analysis"] else "none (answer correct or undetermined)"
    guidance = {"low": "worked example and re-teaching of the concept",
                "medium": "scaffolded prompting with a guiding question",
                "high": "a minimal hint and self-explanation"}[case["band"]]
    return (
        "Generate adaptive tutoring feedback for the student's latest turn.\n"
        f"Problem: {case.get('problem') or 'not provided'}\nReference answer: {case['reference']}\n"
        f"Dialogue so far:\n{case['history']}\n\nStudent turn: {case['student_text']}\n"
        f"Verified correctness y (1 correct, 0 incorrect, None undetermined): {case['y']}\n"
        f"Mastery estimates: {mastery}\n"
        f"Weakest knowledge component: {short_kc(case['target_kc'])} (mastery band: {case['band']}; use {guidance})\n"
        f"Error analysis: {diagnosis}\nPlanned next knowledge component: {short_kc(case['next_kc'])}\n\n"
        "Do not reveal the final answer. Return JSON with exactly these keys: feedback_text, scaffolding_question, "
        "mastery_adaptation, pedagogical_strategy, next_step_hint."
    )

def validate_feedback(obj):
    if not obj or not all(isinstance(obj.get(k), str) and obj[k].strip() for k in FEEDBACK_KEYS):
        return None
    return {k: obj[k].strip() for k in FEEDBACK_KEYS}

def rule_based_feedback(case):
    # Port of kse2026_code/pedagogy.py FeedbackGenerator.rule_based
    target, band = short_kc(case["target_kc"], 80), case["band"]
    strategy = {"low": "worked example and re-teaching of the concept",
                "medium": "scaffolded prompting with a guiding question",
                "high": "minimal hint and self-explanation"}[band]
    if case["y"] == 1:
        text, question = f"Correct. You handled {target} well in this step.", "Can you explain why each step is valid?"
    elif case["y"] == 0:
        explanation = (case["error_analysis"] or {}).get("explanation", "")
        text, question = f"Not quite. {explanation} Look again at the step involving {target}.", "What is the very next step after your last line, and why?"
    else:
        text, question = "I see your reasoning so far. Let us make it concrete with a value.", "What number do you get at the end of that step?"
    return {"feedback_text": text, "scaffolding_question": question,
            "mastery_adaptation": f"Mastery of the weakest KC is in the {band} band.",
            "pedagogical_strategy": strategy, "next_step_hint": f"Next we will work on {short_kc(case['next_kc'], 80)}."}

def leaks_answer(feedback, reference, y):
    if y == 1 or reference is None:
        return float("nan")
    target = to_fraction(reference)
    text = " ".join(feedback[k] for k in ["feedback_text", "scaffolding_question", "next_step_hint"])
    return float(any(to_fraction(x) == target for x in re.findall(NUMBER, text)))

def band_matches(strategy, band):
    return float(any(word in strategy.lower() for word in BAND_KEYWORDS[band]))

def judge_request(case, feedback):
    prompt = (
        "Rate the tutor feedback below on three criteria, each an integer from 1 (poor) to 5 (excellent).\n"
        "correctness: the mathematics and the judgement of the student's answer are right.\n"
        "relevance: the feedback addresses this student's turn, error and weakest skill.\n"
        "clarity: the feedback is clear and appropriate for the learner.\n\n"
        f"Problem: {case.get('problem')}\nReference answer: {case['reference']}\n"
        f"Dialogue so far:\n{case['history'][-1500:]}\nStudent turn: {case['student_text']}\n"
        f"Feedback: {json.dumps(feedback)}\n\n"
        'Return JSON: {"correctness": int, "relevance": int, "clarity": int, "rationale": str}'
    )
    return {"model": JUDGE_MODEL, "temperature": 0, "response_format": {"type": "json_object"},
            "messages": [{"role": "system", "content": "You are an expert mathematics teacher evaluating tutoring feedback."},
                         {"role": "user", "content": prompt}]}

def call_judge(request):
    scores = parse_json_object(openrouter_post(request)) or {}
    return {k: scores.get(k) if isinstance(scores.get(k), int) and 1 <= scores.get(k) <= 5 else None
            for k in ["correctness", "relevance", "clarity"]}

print("Feedback Generator definitions ready.")

In [ ]:
if RUN_FEEDBACK and not integrated_df.empty:
    frame = load_split_frame("mathdial", "test")
    with in_repo():
        context = {(t["dialogue"], t["turn_id"]): t for t in labelled_turns("mathdial", frame, with_context=True)}
    analysed = {(r.dialogue, int(r.turn_id)): r for r in ea_df[ea_df.set == "mathdial_sava_gated"].itertuples()} \
        if not ea_df.empty else {}
    rng = np.random.default_rng(PRIMARY_SEED)
    cases = []
    for verdict in ["correct", "incorrect", "undetermined"]:
        pool = integrated_df[integrated_df.sava_verdict == verdict]
        if verdict == "incorrect":
            pool = pool[[(str(d), int(t)) in analysed for d, t in zip(pool.dialogue, pool.turn_id)]]
        take = min(FEEDBACK_CASES_PER_VERDICT, len(pool))
        chosen = pool.iloc[sorted(rng.choice(len(pool), size=take, replace=False))] if take else pool.iloc[:0]
        for _, g in chosen.iterrows():
            key = (str(g.dialogue), int(g.turn_id))
            turn, mastery = context[key], json.loads(g.kc_mastery_before)
            target = min(mastery, key=mastery.get)
            diagnosis = analysed.get(key)
            cases.append({**turn, "verdict": verdict, "y": {"correct": 1, "incorrect": 0}.get(verdict),
                          "mastery": mastery, "target_kc": target, "band": mastery_band(mastery[target]),
                          "next_kc": g.planner_next_kc,
                          "error_analysis": {"error_category": diagnosis.category, "explanation": diagnosis.explanation}
                          if diagnosis is not None else None})
        print(f"{verdict}: {take} cases (requested {FEEDBACK_CASES_PER_VERDICT}, available {len(pool)})")

    if PEDAGOGY_BACKEND == "local" and GEN_MODEL is None and cases:
        GEN_MODEL, GEN_TOKENIZER, GEN_ADAPTER_NAME = load_generator()
    outputs = generate_texts([feedback_prompt(c) for c in cases]) if cases else []
    records, judge_requests = [], []
    for case, text in zip(cases, outputs):
        parsed = validate_feedback(parse_json_object(text))
        feedback = parsed or rule_based_feedback(case)
        records.append({"dialogue": case["dialogue"], "turn_id": case["turn_id"], "verdict": case["verdict"],
                        "band": case["band"], "valid_json": parsed is not None,
                        "source": "llm" if parsed else "rule_fallback",
                        "answer_leak": leaks_answer(feedback, case["reference"], case["y"]),
                        "strategy_band_match": band_matches(feedback["pedagogical_strategy"], case["band"]),
                        "feedback_words": len(feedback["feedback_text"].split()), **feedback, "raw_output": str(text)[:4000]})
        judge_requests.append(judge_request(case, feedback))
    feedback_df = pd.DataFrame(records)

    if RUN_FEEDBACK_JUDGE and len(feedback_df):
        assert openrouter_key(), "RUN_FEEDBACK_JUDGE=True needs OPENROUTER_API_KEY in Colab Secrets (or set it to False)."
        if PEDAGOGY_BACKEND == "openrouter" and PEDAGOGY_OPENROUTER_MODEL == JUDGE_MODEL:
            print("WARNING: the judge model also generated the feedback; scores carry self-preference bias.")
        scores = [call_judge(request) for request in judge_requests]
        feedback_df = pd.concat([feedback_df, pd.DataFrame(scores)], axis=1)
    else:
        with (RUN_ROOT / "feedback_judge_requests.jsonl").open("w") as handle:
            for request in judge_requests:
                handle.write(json.dumps(request) + "\n")
    feedback_df.to_csv(RUN_ROOT / "feedback_generator_cases.csv", index=False)

    aggregations = {"n": ("valid_json", "size"), "valid_json_rate": ("valid_json", "mean"),
                    "answer_leak_rate": ("answer_leak", "mean"), "strategy_band_match": ("strategy_band_match", "mean"),
                    "mean_feedback_words": ("feedback_words", "mean")}
    for criterion in ["correctness", "relevance", "clarity"]:
        if criterion in feedback_df.columns:
            aggregations[f"{criterion}_mean"] = (criterion, "mean")
            aggregations[f"{criterion}_sd"] = (criterion, "std")
    feedback_summary_df = pd.concat([feedback_df.groupby("verdict").agg(**aggregations),
                                     feedback_df.assign(verdict="all").groupby("verdict").agg(**aggregations)])
    feedback_summary_df.to_csv(RUN_ROOT / "feedback_generator_summary.csv")
    display(feedback_summary_df)
    display(feedback_df[["verdict", "band", "source", "pedagogical_strategy", "feedback_text"]].head(10))

if GEN_MODEL is not None:
    GEN_MODEL = None
    gc.collect()
    torch.cuda.empty_cache()

## 17. Integrity checks, manifest and download

The run is accepted only when every enabled stream produced artifacts of the expected size. The manifest records sample counts per split, preprocessing failures, data-file hashes, library versions, thresholds and golden-test results, so two runs can be checked for silent data drift before they are compared.

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

expected_models = len(DATASETS) * len(MODEL_RUNS)
assert len(golden_df) == expected_models, (len(golden_df), expected_models)
assert len(metrics_df[metrics_df.split == "test"]) == expected_models
if RUN_CCMF:
    for dataset in DATASETS:
        for model_key, seed in CCMF_RUNS:
            variants = set(ccmf_df[(ccmf_df.dataset == dataset) & (ccmf_df.model == model_key) &
                                   (ccmf_df.seed == seed_tag(model_key, seed))].variant)
            required = {"mean_raw", "mean_platt", "noisy_and_platt", "ccmf_no_update", "ccmf_model_update", "ccmf_gold_update"}
            required |= {"ccmf_sava_update"} if dataset == "mathdial" else set()
            assert variants == required, (dataset, model_key, variants)
if RUN_SAVA and "mathdial" in DATASETS:
    assert len(sava_benchmark_df) == 12 * SAVA_N_REFERENCES
if RUN_PLANNER:
    assert len(planner_df) == len(MAIN_POLICIES) and len(planner_grid_df) == 144 * len(MAIN_POLICIES)
if RUN_INTEGRATED and RUN_CCMF and RUN_SAVA and "mathdial" in DATASETS:
    model_key, seed = INTEGRATED_TARGET
    assert len(integrated_df) == len(load_rows("mathdial", model_key, seed, "test"))
if RUN_ERROR_ANALYZER:
    assert not ea_df.empty, "Error Analyzer produced no predictions."
    assert set(ea_df.category) <= set(ERROR_CLASSES)
if RUN_FEEDBACK and not integrated_df.empty:
    assert len(feedback_df) > 0 and set(feedback_df.verdict) <= {"correct", "incorrect", "undetermined"}

failures = {}
for log in RUN_ROOT.glob("*/*/*/test/command.log"):
    found = re.findall(r"(\d+) / (\d+) dialogues failed processing", log.read_text())
    failures[str(log.parent.relative_to(RUN_ROOT))] = [[int(a), int(b)] for a, b in found]
split_sizes = {f"{d}/{m}/{seed_tag(m, s)}/{split}": len(load_rows(d, m, s, split))
               for d in DATASETS for m, s in MODEL_RUNS for split in ["val", "test"]}
data_hashes = {p.name: sha256(p) for p in sorted((REPO / "data/annotated").glob("*.csv"))}
ccmf_parameters = {p.stem: json.loads(p.read_text())["parameters"] for p in RUN_ROOT.glob("ccmf_parameters_*.json")}

manifest = {
    "run_id": RUN_ID, "smoke": SMOKE, "seeds": SEEDS, "base_model": BASE_MODEL,
    "repository_commit": REPO_COMMIT, "dependency_stack": DEPENDENCY_STACK, "versions": VERSIONS,
    "datasets": DATASETS, "model_runs": [[m, seed_tag(m, s)] for m, s in MODEL_RUNS],
    "adapters": ADAPTER_PROVENANCE, "artifact_store": ARTIFACT_STORE, "adapter_source": ADAPTER_SOURCE,
    "comta_mode": COMTA_MODE, "fit_on": FIT_ON,
    "split_sizes": split_sizes, "preprocessing_failures": failures, "data_sha256": data_hashes,
    "golden_test": golden_df.to_dict(orient="records"),
    "thresholds": threshold_df[["dataset", "model", "seed", "threshold"]].to_dict(orient="records"),
    "ccmf_parameters": ccmf_parameters,
    "planner_config": asdict(PAPER_CONFIG), "planner_eval_seeds": [EVAL_SEEDS[0], EVAL_SEEDS[-1]],
    "pedagogy": {"backend": PEDAGOGY_BACKEND,
                 "generator": BASE_MODEL if PEDAGOGY_BACKEND == "local" else f"openrouter:{PEDAGOGY_OPENROUTER_MODEL}",
                 "decoding": "4-bit NF4, greedy" if PEDAGOGY_BACKEND == "local" else "API, temperature 0",
                 "kt_adapter": GEN_ADAPTER_NAME,
                 "error_analyzer_items": ea_df.groupby("set").size().to_dict() if not ea_df.empty else {},
                 "error_analyzer_valid_json_rate": float(ea_df.valid_json.mean()) if not ea_df.empty else None,
                 "error_analyzer_gold_supplied": (WORK / "error_analyzer_gold.csv").exists(),
                 "feedback_cases": int(len(feedback_df)),
                 "feedback_judge": f"openrouter:{JUDGE_MODEL}" if RUN_FEEDBACK_JUDGE else None},
    "evidence_labels": {"qlora": "REAL DATA / SINGLE-FOLD REPLICATION", "ccmf": "REAL-LOGIT ABLATION",
                        "sava": "CONSTRUCTED BENCHMARK + REAL-TURN AGREEMENT", "planner": "SIMULATED DATA",
                        "integrated": "REAL DATA / MATHDIAL TEST TURNS",
                        "error_analyzer": "REAL DATA / LLM AGENT (scored only against supplied human gold)",
                        "feedback": "REAL DATA / LLM AGENT (automatic checks; LLM-as-judge only if enabled)"},
}
(RUN_ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

archive = shutil.make_archive(str(RUN_ROOT), "zip", RUN_ROOT.parent, RUN_ROOT.name)
print("Validated result bundle:", archive)
if ARTIFACT_STORE != "none":
    store_save_results(archive)
if SMOKE:
    print("SMOKE RUN PASSED. Set SMOKE = False in Section 1 and choose Runtime > Run all for the full experiment.")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download manually:", archive)

## Reporting rules

1. Report only full runs (`"smoke": false` in `manifest.json`).
2. Compare QLoRA with zero-shot only when dataset, split, sample count (`split_sizes`), aggregation and seed match; the golden test must have passed.
3. Report validation-selected threshold results separately from the released fixed-0.5 protocol.
4. CCMF: report `ccmf_sava_update` as the deployable form of Eq. (6) and `ccmf_gold_update` only as an upper bound; quote paired `delta_ci` intervals, not overlapping marginal intervals.
5. SAVA is a verify-or-abstain component: report coverage and accuracy-when-spoken together with balanced accuracy, the `always_correct` and `numeric_regex` baselines, the stage table, and the real-turn agreement split by final and non-final turns. Do not tune extraction rules on the constructed benchmark; a rule change must be justified on real-turn agreement.
6. Planner: label as SIMULATED DATA; report the main table, the ablation, and the grid win counts including the filtered-curriculum baselines.
7. Do not claim that QLoRA validates SAVA, CCMF, Error Analyzer, Feedback Generator or Planner; they are distinct evidence streams.
8. Error Analyzer: report metrics only from `error_analyzer_metrics.csv` (human gold annotated blind to predictions), with the valid-JSON rate, the rule-based and majority baselines, and the definition of relaxed agreement. The number of CoMTA turns is whatever the run reports, not a fixed 46.
9. Feedback Generator: report the automatic checks (validity, answer leakage, strategy–band match); report judge scores only when the judge was enabled, naming the judge model and stating that it is an automated proxy.
10. CoMTA (evaluation only): report zero-shot and MathDial-adapter results with MathDial-fitted thresholds and CCMF parameters; never report numbers from `COMTA_MODE = "train"` without Khan Academy permission.
11. Preserve the full result ZIP and `manifest.json` with the submission artifacts.